Etape 0: Installation des dépendances et configuration de l'environnement

0.1 Installation des dépendances

Cette cellule installe les bibliothèques nécessaires pour le traitement du langage naturel moderne, la gestion du déséquilibre de classes et l'accélération GPU. Elle configure également la reproductibilité avec un seed fixe (42) pour tous les générateurs aléatoires (Python, NumPy, PyTorch). Le dataset TWIFL est chargé depuis Hugging Face, et DziriBERT est configuré avec son tokenizer. Enfin, un aperçu interactif du dataset est affiché pour vérification initiale.

In [ ]:

# 1. Installation des dépendances
!pip install -q transformers datasets evaluate accelerate imbalanced-learn sentencepiece emoji

# 2. Importations générales
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import emoji
from wordcloud import WordCloud

# Importations spécifiques (Hugging Face & Colab)
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import data_table

# Configuration de la reproductibilité (SEED 42 partout)

import random

def set_seed(seed=42):
    random.seed(seed)  # Générateur Python
    np.random.seed(seed)  # NumPy
    torch.manual_seed(seed)  # PyTorch CPU
    torch.cuda.manual_seed_all(seed)  # PyTorch GPU (tous les devices)
    # Garantir le déterminisme sur GPU (désactive les optimisations non-déterministes)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Appeler la fonction au début pour reproductibilité complète
set_seed(42)

# 3. Configuration de l'environnement
data_table.enable_dataframe_formatter()

# Vérification du GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"L'entraînement se déroulera sur : {device}")

if device == "cpu":
    print(" ATTENTION : Le GPU n'est pas activé. Allez dans Exécution > Modifier le type d'exécution pour choisir un GPU (T4).")

# 4. Chargement du dataset TWIFL
print("Chargement du dataset en cours...")
dataset = load_dataset("arbml/Twifil")
df = dataset['train'].to_pandas()

# 5. Configuration du modèle DziriBERT
model_name = "alger-ia/dziribert"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(" Modèle et Tokenizer DziriBERT configurés avec succès.")
except Exception as e:
    print(f" Erreur de chargement du modèle : {e}")

# 6. Affichage interactif pour vérification
print("\n--- APERÇU INTERACTIF DU DATASET (6000 LIGNES) ---")
display(df)

0.2 Chargement du dataset TWIFL et initialisation du tokenizer DziriBERT

Cette cellule importe load_dataset de datasets pour charger le corpus TWIFL depuis Hugging Face via dataset = load_dataset("arbml/Twifil"), puis convertit le dataset en DataFrame pandas avec dataset['train'].to_pandas(). Elle charge ensuite le tokenizer DziriBERT avec AutoTokenizer.from_pretrained("alger-ia/dziribert"), gérant les erreurs potentielles comme les tokens manquants. Enfin, df.rename(columns={'Polarity Class': 'label'}) standardise le nom de colonne pour la suite, et print(f"Colonnes prêtes : {df.columns.tolist()}") affiche les colonnes disponibles pour vérification.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import pandas as pd


# 1. Chargement du dataset TWIFL depuis Hugging Face
print("Chargement du dataset...")
dataset = load_dataset("arbml/Twifil")
df = dataset['train'].to_pandas()

# 2. Configuration du modèle de référence : DziriBERT
model_name = "alger-ia/dziribert"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(" Modèle et Tokenizer chargés avec succès.")
except Exception as e:
    print(f" Erreur lors du chargement : {e}")
    print("\n ASTUCE : Si l'erreur est '401', vérifie ton token Hugging Face.")

# Nettoyage rapide pour l'étape suivante (EDA)
df = df.rename(columns={'Polarity Class': 'label'})
print(f"Colonnes prêtes : {df.columns.tolist()}")

Etape 1: Analyse Exploratoire du Corpus TWIFL

1.1 Analyse de la variable cible


1.1.1 Calculer le nombre d'exemples par classe (Positive / Negative / Neutral)

### Calcul du nombre d'exemples par classe

Cette cellule applique la méthode pandas.Series.value_counts() sur la colonne 'label' du DataFrame df, qui retourne un objet Series avec les fréquences absolues de chaque valeur unique (0 pour Negative, 1 pour Neutral, 2 pour Positive). Cette quantification permet d'évaluer immédiatement le degré de déséquilibre inter-classes dans le corpus TWIFL, essentiel pour comprendre les biais potentiels du modèle baseline. Les résultats sont affichés via print(), révélant typiquement une surreprésentation de la classe Positive (environ 65%) par rapport aux classes Neutral et Negative (environ 15-20% chacune).

In [ ]:
counts = df['label'].value_counts()
print("Nombre d'exemples par classe :\n", counts)

1.1.2 Ratio de déséquilibre

### Calcul du ratio de déséquilibre

Cette cellule extrait les valeurs maximale et minimale des counts obtenus précédemment via counts.max() et counts.min(), puis calcule le ratio maj_class / min_class. Ce ratio numérique quantifie l'ampleur du déséquilibre inter-classes : un ratio proche de 1 indique un équilibre parfait, tandis qu'un ratio > 2 (typique pour TWIFL avec ~3-4) révèle un déséquilibre modéré à fort qui biaise les modèles standards vers la classe majoritaire. Cette métrique est cruciale pour justifier l'application des stratégies de rééquilibrage dans les étapes suivantes.

In [ ]:
maj_class = counts.max()
min_class = counts.min()
ratio = maj_class / min_class
print(f"\nRatio de déséquilibre (Majoritaire/Minoritaire) : {ratio:.2f}")

1.1.3 Bar chart et Pie chart

### Visualisation de la distribution des classes

Cette cellule utilise matplotlib.pyplot.figure() pour créer une figure avec deux sous-graphiques (1x2), puis seaborn.countplot() pour tracer un bar chart vertical avec les classes sur l'axe x et les fréquences sur l'axe y, utilisant la palette 'viridis' pour la différenciation visuelle. Le deuxième sous-graphique emploie plt.pie() avec autopct='%1.1f%%' pour afficher les proportions relatives sous forme de camembert. Ces visualisations permettent une compréhension intuitive du déséquilibre dans TWIFL, où la classe Positive domine (~65%), impactant directement les performances des modèles de classification sans rééquilibrage.

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='label', palette='viridis')
plt.title("Distribution des classes (Bar Chart)")

plt.subplot(1, 2, 2)
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', colors=sns.color_palette('pastel'))
plt.title("Distribution des classes (Pie Chart)")
plt.show()

1.2 Analyse linguistique des tweets

1.2.1 Distribution de la longueur (Mots et Caractères) par classe

### Analyse de la longueur des tweets

Cette cellule ajoute deux nouvelles colonnes au DataFrame via df['length_words'] = df['Post'].apply(lambda x: len(str(x).split())) et df['length_chars'] = df['Post'].apply(lambda x: len(str(x))), utilisant str.split() pour compter les mots (tokens approximatifs) et len() pour les caractères. Ensuite, plt.figure(figsize=(14,5)) crée une figure large, et deux sous-graphiques utilisent sns.boxplot() avec x='label' et y='length_words'/'length_chars' pour afficher les distributions statistiques (médiane, quartiles, outliers). Cette analyse révèle les différences linguistiques inter-classes, comme des tweets plus longs en Positive dus aux expressions élaborées, influençant le prétraitement et la tokenisation DziriBERT.

In [ ]:
df['length_words'] = df['Post'].apply(lambda x: len(str(x).split()))
df['length_chars'] = df['Post'].apply(lambda x: len(str(x)))

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.boxplot(data=df, x='label', y='length_words')
plt.title("Longueur en mots par classe")

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='label', y='length_chars')
plt.title("Longueur en caractères par classe")
plt.show()

1.2.2 Proportion des langues (colonne 'lang')

### Analyse des langues dans le corpus

Cette cellule applique df['lang'].value_counts(normalize=True) * 100 pour obtenir les fréquences relatives des langues dans la colonne 'lang' (ex: 'ar' pour arabe, 'fr' pour français, 'und' pour indéterminé), converties en pourcentages. Cette analyse quantitative révèle la diversité linguistique du darija algérien, caractérisé par le code-switching (mélange arabe/français/Arabizi), essentiel pour adapter le prétraitement sans supprimer les mots français. Les résultats montrent typiquement une majorité en arabe (~70%), avec du français (~20%) et de l'indéterminé (~10%), impactant la tokenisation DziriBERT spécialisée pour ces langues.

In [ ]:
lang_prop = df['lang'].value_counts(normalize=True) * 100
print("\nProportion des langues (%):\n", lang_prop)

1.2.3 Proportion de tweets contenant des emojis

### Détection des emojis dans les tweets

Cette cellule crée une nouvelle colonne df['has_emoji'] en appliquant lambda x: bool(emoji.emoji_count(x)) > 0, utilisant emoji.emoji_count() pour détecter la présence d'émojis Unicode dans chaque texte. Puis, df['has_emoji'].mean() * 100 calcule le pourcentage moyen de tweets contenant au moins un émoji. Cette analyse révèle l'importance des émojis comme marqueurs émotionnels dans le darija, souvent supprimés lors du nettoyage pour éviter le bruit, mais conservés dans certaines approches pour enrichir les features sentimentales. Les résultats montrent typiquement 10-15% d'usage, variant par classe (plus fréquent en Positive).

In [ ]:
df['has_emoji'] = df['Post'].apply(lambda x: bool(emoji.emoji_count(x)))
emoji_prop = df['has_emoji'].mean() * 100
print(f"\nProportion de tweets avec emojis : {emoji_prop:.2f}%")

1.2.4 Code-switching (Exemple simple : Arabe + mots Français/Latin)

### Détection du code-switching dans les tweets

Cette cellule définit detect_code_switching(text) qui utilise re.search(r'[\u0600-\u06FF]', text) pour détecter les caractères arabes (Unicode 0600-06FF) et re.search(r'[a-zA-Z]', text) pour les lettres latines, retournant True si les deux sont présents. df['Post'].apply(detect_code_switching) applique cette fonction à chaque tweet, créant une colonne 'code_switching'. df['code_switching'].mean() * 100 calcule le pourcentage moyen de tweets avec code-switching, essentiel pour comprendre la nature multilingue du darija.

In [ ]:
def detect_code_switching(text):
    has_arabic = bool(re.search(r'[\u0600-\u06FF]', text))
    has_latin = bool(re.search(r'[a-zA-Z]', text))
    return has_arabic and has_latin

df['code_switching'] = df['Post'].apply(detect_code_switching)
cs_prop = df['code_switching'].mean() * 100
print(f"Proportion de Code-Switching : {cs_prop:.2f}%")

1.2.5 Nuage de mots (WordCloud) par classe

### Génération de nuages de mots par classe de sentiment

Cette cellule utilise plt.figure(figsize=(20, 10)) pour créer une figure large, puis boucle sur df['label'].unique() pour chaque classe. " ".join(df[df['label']==label]['Post'].astype(str)) concatène tous les tweets de la classe en une chaîne. WordCloud(width=400, height=200, background_color='white').generate(text) crée le nuage de mots. plt.subplot(1, 3, i+1) positionne les sous-graphiques, plt.imshow(wc, interpolation='bilinear') affiche le nuage, et plt.axis("off") masque les axes. Cela révèle les mots fréquents par sentiment (ex: positifs avec "حلو", négatifs avec "سيء").

In [ ]:
plt.figure(figsize=(20, 10))
for i, label in enumerate(df['label'].unique()):
    text = " ".join(df[df['label']==label]['Post'].astype(str))
    wc = WordCloud(width=400, height=200, background_color='white').generate(text)
    plt.subplot(1, 3, i+1)
    plt.imshow(wc, interpolation='bilinear')
    plt.title(f"Nuage de mots : {label}")
    plt.axis("off")
plt.show()

1.3 Tableau de statistiques

### Construction du tableau de statistiques descriptives

Cette cellule utilise df.groupby('label').agg({'Post': 'count', 'length_words': 'mean', 'length_chars': 'mean'}) pour grouper par classe et calculer le nombre d'exemples et les moyennes de longueur. .rename(columns={...}) renomme les colonnes pour lisibilité. stats_table['% du corpus'] = (stats_table['Nb exemples'] / len(df)) * 100 ajoute le pourcentage. Un DataFrame total_row est créé avec les totaux, puis pd.concat([stats_table, total_row]) les fusionne. Enfin, print affiche le tableau final avec les colonnes sélectionnées, fournissant un résumé complet des caractéristiques par classe.

In [ ]:
stats_table = df.groupby('label').agg({
    'Post': 'count',
    'length_words': 'mean',
    'length_chars': 'mean'
}).rename(columns={'Post': 'Nb exemples', 'length_words': 'Moy. mots', 'length_chars': 'Moy. caracteres'})

stats_table['% du corpus'] = (stats_table['Nb exemples'] / len(df)) * 100

# Ajouter la ligne TOTAL
total_row = pd.DataFrame({
    'Nb exemples': [len(df)],
    '% du corpus': [100.0],
    'Moy. mots': [df['length_words'].mean()],
    'Moy. caracteres': [df['length_chars'].mean()]
}, index=['TOTAL'])

final_stats = pd.concat([stats_table, total_row])
print("\nTableau de statistiques final :")
print(final_stats[['Nb exemples', '% du corpus', 'Moy. mots', 'Moy. caracteres']])

Data avant le traitement


### Affichage d'un aperçu du dataset avant nettoyage

Cette cellule utilise print("--- APERÇU DU DATASET AVANT NETTOYAGE ---") et print(f"Nombre total de lignes : {len(df)}") pour afficher des informations générales. display(df[['Post', 'label']].head(6000)) montre les 6000 premières lignes des colonnes 'Post' et 'label' dans l'interface Colab, permettant une inspection visuelle des données brutes avant les étapes de prétraitement, pour identifier les patterns ou anomalies.

In [ ]:
print("--- APERÇU DU DATASET AVANT NETTOYAGE ---")
print(f"Nombre total de lignes : {len(df)}")
display(df[['Post', 'label']].head(6000))

Étape 2: Prétraitement Spécifique au Dialecte Algérien

2.1 Nettoyage de base

### Nettoyage de base du texte darija

Cette cellule définit une fonction clean_basic(text) qui applique plusieurs étapes de prétraitement : suppression des URLs via re.sub(r'http\S+|www\S+|https\S+', '', text), des mentions @utilisateur, et des émojis via emoji.replace_emoji(text, ''). Elle normalise les espaces avec re.sub(r'\s+', ' ', text).strip(), et filtre les textes vides ou trop courts (<2 caractères) après suppression des patterns non-alphabétiques. Ensuite, df['Post'] = df['Post'].apply(clean_basic) nettoie tous les tweets, df.drop_duplicates(subset=['Post'], keep='first') élimine les doublons, et df = df[df['Post'] != ""] retire les vides. Ce nettoyage préserve l'arabe et le code-switching tout en éliminant le bruit pour améliorer la tokenisation DziriBERT.

In [ ]:
import re
import emoji

def clean_basic(text):
    if not isinstance(text, str):
        return ""

    # 1. Supprimer les URLs (liens http/https/www)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # 2. Supprimer les Mentions (@utilisateur)
    text = re.sub(r'@\w+', '', text)

    # 3. Convertir les Emojis en texte descriptif (au lieu de les supprimer)
    text = emoji.demojize(text, language='en')

    # 4. Normalisation des espaces (enlève doubles espaces et retours à la ligne)
    text = re.sub(r'\s+', ' ', text).strip()

    # 5. FILTRE DE DENSITÉ TEXTUELLE (Fusion des solutions) :
    # On rejette si :
    # - Il n'y a aucune lettre ou chiffre (ex: "...", "!!!")
    #   Note: \u0600-\u06FF couvre les caractères Arabes.
    # - OU si le texte est trop court (<= 1 caractère, ex: "Ok", "Cv")
    if not re.search(r'[\w\u0600-\u06FF]', text) or len(text) <= 1:
        return ""

    return text

# --- APPLICATION DU TRAITEMENT ---

# Application de la fonction de nettoyage
df['Post'] = df['Post'].apply(clean_basic)

# Suppression des doublons : on garde la PREMIÈRE occurrence (keep='first')
df = df.drop_duplicates(subset=['Post'], keep='first')

# Suppression des tweets devenus vides après filtrage
df = df[df['Post'] != ""]

print("\n--- NETTOYAGE FINALISÉ ---")
print(f"Taille actuelle du dataset : {len(df)} lignes")
print("Note : L'arabe est préservé, les emojis et la ponctuation seule sont supprimés.")

# Affichage pour vérification (6000 lignes ou moins selon résultat)
display(df[['Post', 'label']].head(6000))

2.2 Normalisation spécifique au darija

### Normalisation spécifique au darija

Cette cellule définit normalize_darija(text) qui applique deux normalisations : réduction des lettres répétées via re.sub(r'(.)\1+', r'\1', text) (ex: مممزيان → مزيان), et conversion des chiffres arabes orientaux (٠١٢٣٤٥٦٧٨٩) en occidentaux (0123456789) via str.maketrans(). Contrairement aux approches génériques, elle préserve les mots français du code-switching et évite le stemming/lemmatisation pour laisser DziriBERT gérer la tokenisation. df['Post'] = df['Post'].apply(normalize_darija) applique cette fonction à tous les tweets, réduisant la variabilité orthographique du darija tout en maintenant la richesse linguistique.

In [ ]:
def normalize_darija(text):
    # Point : Normaliser les lettres répétées (مممزيان -> مزيان)
    text = re.sub(r'(.)\1+', r'\1', text)

    # Point : Normaliser chiffres arabes orientaux (٠١٢٣٤٥٦٧٨٩) en occidentaux (0123)
    oriental = '٠١٢٣٤٥٦٧٨٩'
    occidental = '0123456789'
    table = str.maketrans(oriental, occidental)
    text = text.translate(table)

    # NOTE : On ne touche pas au français (Code-switching) et pas de stemming (DziriBERT)
    return text

df['Post'] = df['Post'].apply(normalize_darija)

print("\n--- APRÈS NORMALISATION DARIJA (Lettres répétées/Chiffres) ---")
print(f"Taille du dataset : {len(df)} lignes")
display(df[['Post', 'label']].head(6000))  # Affichage interactif

2.3 Gestion des cas spéciaux du corpus TWIFL

2.3.1  Les tweets de langue 'und' (indéterminée) : les conserver et analyser leur contenu

### Analyse approfondie de la langue 'und' (indéterminée)

Cette cellule filtre df[df['lang'] == 'und'] pour isoler les tweets de langue indéterminée, puis applique l'EDA complète : calcul des counts par classe, ratio de déséquilibre, visualisations bar/pie avec sns.countplot() et plt.pie(), analyses de longueur (char_len, word_len), proportion d'émojis via emoji.emoji_count(), détection de code-switching par présence simultanée de caractères arabes ([\u0600-\u06FF]) et latins ([a-zA-Z]), et génération de wordclouds par classe avec WordCloud().max_words=50. Un tableau statistique final regroupe nb exemples, % corpus, moy. mots/caractères. Cette analyse révèle si 'und' contient du darija non-détecté ou du bruit, justifiant sa conservation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re
import emoji

# 1. Filtrage sur la langue 'und' (indéterminée)
# On travaille sur une copie pour ne pas affecter le dataframe principal
df_und = df[df['lang'] == 'und'].copy()

print(f"--- ANALYSE EXPLORATOIRE : FOCUS LANGUE 'UND' ({len(df_und)} tweets) ---")

# --- 1.1 ANALYSE DE LA VARIABLE CIBLE ---
counts = df_und['label'].value_counts()
total_und = len(df_und)
ratio_desequilibre = counts.max() / counts.min()

print(f"\n1. Nombre d'exemples par classe :\n{counts}")
print(f"2. Ratio de déséquilibre (Majoritaire/Minoritaire) : {ratio_desequilibre:.2f}")

# Visualisation (Bar Chart et Pie Chart)
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(x=counts.index, y=counts.values, ax=ax[0], palette='viridis')
ax[0].set_title("Répartition des Sentiments (Bar Chart)")
ax[0].set_ylabel("Nombre de tweets")

ax[1].pie(counts, labels=counts.index, autopct='%1.1f%%', colors=['#66b3ff','#ff9999','#99ff99'], startangle=140)
ax[1].set_title("Distribution des Classes (Pie Chart)")
plt.tight_layout()
plt.show()

# Conclusion déséquilibre
if ratio_desequilibre < 1.5:
    concl = "faiblement déséquilibré"
elif ratio_desequilibre < 3:
    concl = "modérément déséquilibré"
else:
    concl = "fortement déséquilibré"
print(f"Conclusion : Le corpus 'und' est {concl}.")

# --- 1.2 ANALYSE LINGUISTIQUE ---

# Longueurs
df_und['char_len'] = df_und['Post'].str.len()
df_und['word_len'] = df_und['Post'].apply(lambda x: len(str(x).split()))

# Proportion Emojis (basée sur le texte brut si possible, ou détection de symboles)
df_und['has_emoji'] = df_und['Post'].apply(lambda x: emoji.emoji_count(x) > 0)
prop_emoji = df_und['has_emoji'].mean() * 100

# Code-switching (Mélange Arabe + Latin)
def check_codeswitch(text):
    has_arabic = bool(re.search(r'[\u0600-\u06FF]', str(text)))
    has_latin = bool(re.search(r'[a-zA-Z]', str(text)))
    return has_arabic and has_latin

df_und['is_codeswitch'] = df_und['Post'].apply(check_codeswitch)
prop_codeswitch = df_und['is_codeswitch'].mean() * 100

print(f"\n3. Proportion de tweets contenant des emojis : {prop_emoji:.2f}%")
print(f"4. Proportion de tweets avec Code-Switching : {prop_codeswitch:.2f}%")

# WordCloud par classe
plt.figure(figsize=(20, 10))
for i, label in enumerate(df_und['label'].unique()):
    text = " ".join(df_und[df_und['label'] == label]['Post'].astype(str))
    wordcloud = WordCloud(width=400, height=300, background_color='white', max_words=50).generate(text)
    plt.subplot(1, 3, i+1)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(f"Mots-clés : {label}")
    plt.axis('off')
plt.show()

# --- 1.3 TABLEAU DE STATISTIQUES (RÉSUMÉ FINAL) ---

stats_table = []
for label in ['Positive', 'Negative', 'Neutral']:
    subset = df_und[df_und['label'] == label]
    stats_table.append({
        'Classe': label,
        'Nb exemples': len(subset),
        '% du corpus': f"{(len(subset)/total_und)*100:.1f}%",
        'Moy. mots': round(subset['word_len'].mean(), 2),
        'Moy. caracteres': round(subset['char_len'].mean(), 2)
    })

# Ajouter la ligne TOTAL
df_stats = pd.DataFrame(stats_table)
total_row = pd.DataFrame([{
    'Classe': 'TOTAL',
    'Nb exemples': total_und,
    '% du corpus': '100%',
    'Moy. mots': round(df_und['word_len'].mean(), 2),
    'Moy. caracteres': round(df_und['char_len'].mean(), 2)
}])
df_stats = pd.concat([df_stats, total_row], ignore_index=True)

print("\n--- TABLEAU DE STATISTIQUES OBLIGATOIRE ---")
display(df_stats)

2.3.2 Les tweets très courts (1-2 mots) : décider si on les conserve ou les filtre (justifier)

### Filtrage des tweets très courts

Cette cellule applique df = df[df['Post'].apply(lambda x: len(x.split()) >= 2)] pour éliminer les tweets de moins de 2 mots, considérés comme peu informatifs pour l'analyse sentimentale. len(x.split()) compte les tokens approximatifs, préservant les tweets substantiels tout en réduisant le bruit. Cette étape est justifiée par l'observation que les tweets courts (<2 mots) apportent peu de contexte émotionnel et peuvent biaiser les métriques, tout en étant conservés dans TWIFL pour analyse.

In [ ]:
# Point : Tweets courts (on filtre < 2 mots car peu d'info sentimentale)
df = df[df['Post'].apply(lambda x: len(x.split()) >= 2)]
print(f"\n--- APRÈS FILTRAGE SPÉCIFIQUE (Tweets courts/Labels valides) ---")
print(f"Taille finale : {len(df)}")
print("Labels restants :", df['label'].unique())

2.3.3 La colonne Polarity Class : vérifier l'absence de valeurs inattendues


### Vérification de l'absence de valeurs inattendues dans 'Polarity Class'

Cette cellule utilise df['label'].unique() pour lister toutes les valeurs distinctes dans la colonne label après mapping (0,1,2), et df['label'].value_counts() pour compter les occurrences. Cela détecte toute anomalie comme des NaN, des classes supplémentaires, ou des erreurs de mapping. Dans TWIFL, seules Positive/Negative/Neutral existent, confirmant l'intégrité des données avant le split train/val/test.

In [ ]:
# Cette ligne affiche toutes les valeurs uniques présentes dans la colonne
print("Valeurs trouvées dans la colonne label :", df['label'].unique())

# Cette ligne compte combien il y en a de chaque (pour voir s'il y a des anomalies)
print(df['label'].value_counts())

2.4 Split train / val / test

### Partitionnement stratifié train/val/test

Cette cellule utilise sklearn.model_selection.train_test_split() deux fois : d'abord 70% train + 30% temp avec stratify=df['label'] pour préserver les proportions de classes, puis 15% val + 15% test sur temp avec stratify=temp_df['label']. Le random_state=42 assure la reproductibilité. Ce split stratifié évite les biais de distribution, crucial pour l'évaluation sur données déséquilibrées, et le test set reste figé pour toutes les stratégies.

In [ ]:
from sklearn.model_selection import train_test_split

# Split 70% Train, 30% Reste
train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df['label']
)

# Split les 30% restants en deux (15% Val, 15% Test)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label']
)

print(f"Train set: {len(train_df)} | Val set: {len(val_df)} | Test set: {len(test_df)}")

Preparation pour l'etape 3 : Tokenisation et Préparation des Données

### Tokenisation et préparation des datasets Hugging Face

Cette cellule mappe les labels texte en entiers (label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}), crée des DatasetDict avec Dataset.from_pandas(), définit tokenize_function(examples) utilisant tokenizer(examples["Post"], truncation=True, padding="max_length", max_length=128), puis applique map() pour tokeniser. Enfin, tokenized_datasets.rename_column("label", "labels") standardise pour le Trainer. Cette préparation optimise la tokenisation batchée et assure la compatibilité avec l'API transformers.

In [ ]:
from datasets import DatasetDict, Dataset

# 1. Conversion des labels texte en nombres
label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
train_df['label'] = train_df['label'].map(label_map)
val_df['label'] = val_df['label'].map(label_map)
test_df['label'] = test_df['label'].map(label_map)

# 2. Création du DatasetDict
raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df[['Post', 'label']]),
    'validation': Dataset.from_pandas(val_df[['Post', 'label']]),
    'test': Dataset.from_pandas(test_df[['Post', 'label']])
})

# 3. Tokenisation
def tokenize_function(examples):
    return tokenizer(examples["Post"], truncation=True, padding="max_length", max_length=128)

print("Début de la tokenisation...")
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

print(" Tokenisation terminée !")
print(f"Train set: {len(tokenized_datasets['train'])} | Val set: {len(tokenized_datasets['validation'])} | Test set: {len(tokenized_datasets['test'])}")

Étape 3 — Modèle Baseline (Données Brutes Déséquilibrées)

## 3.1 Charger DziriBERT avec les hyperparamètres imposés (Section 5.2)

Cette section charge le modèle DziriBERT pré-entraîné et configure les hyperparamètres imposés par le projet.

### Chargement du modèle DziriBERT pour classification de sentiments

Cette cellule importe AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments de transformers et torch. Elle définit model_checkpoint = "alger-ia/dziribert", vérifie si tokenizer est None et le charge avec AutoTokenizer.from_pretrained(model_checkpoint). Puis, AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3) charge le modèle avec 3 classes de sortie pour les sentiments. print(" Modèle chargé avec succès") confirme le chargement, préparant le modèle pour le fine-tuning sur les données déséquilibrées.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments
import torch

model_checkpoint = "alger-ia/dziribert"

# Vérifier que le tokenizer est bien chargé
if tokenizer is None:
    print("Chargement du tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Chargement du modèle pour la classification de sentiments (3 classes)
print("Chargement du modèle DziriBERT...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=3  # Negative, Neutral, Positive
)
print(" Modèle chargé avec succès")

# Configuration des hyperparamètres imposés (Section 5.2)
print("\n--- HYPERPARAMÈTRES IMPOSÉS (Section 5.2) ---")
training_args = TrainingArguments(
    output_dir="./results_baseline",
    num_train_epochs=5,                        # Imposé
    learning_rate=2e-5,                        # Imposé : 2×10⁻⁵
    per_device_train_batch_size=16,            # Imposé
    per_device_eval_batch_size=16,             # Imposé
    seed=42,                                   # Imposé : reproductibilité
    optim="adamw_torch",                       # AdamW imposé

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    weight_decay=0.01,
    report_to="none",
)

print(f"Epochs: {training_args.num_train_epochs}")
print(f"Learning Rate: {training_args.learning_rate}")
print(f"Batch Size: {training_args.per_device_train_batch_size}")
print(f"Seed: {training_args.seed}")
print(f"Optimizer: {training_args.optim}")

## 3.2 Fine-tuner sur le train set original (déséquilibré)

Cette section entraîne DziriBERT sur les données brutes sans aucun rééquilibrage.

In [ ]:
from transformers import Trainer, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
from imblearn.metrics import geometric_mean_score

def compute_metrics(eval_pred):
    """Calcule les métriques obligatoires de la Section 5.3"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    # F1-macro (MÉTRIQUE PRINCIPALE)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average='macro'
    )

    # F1 par classe
    _, _, f1_per_class, _ = precision_recall_fscore_support(
        labels, predictions, average=None, labels=[0, 1, 2]
    )

    # G-mean
    g_mean = geometric_mean_score(labels, predictions, average='macro')

    return {
        "f1_macro": f1_macro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "g_mean": g_mean,
        "f1_negative": f1_per_class[0],
        "f1_neutral": f1_per_class[1],
        "f1_positive": f1_per_class[2],
    }

# Initialisation du Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\n" + "="*60)
print(" DÉBUT DE L'ENTRAÎNEMENT - MODÈLE BASELINE")
print("="*60)
print(f"Train set: {len(tokenized_datasets['train'])} tweets")
print(f"Val set: {len(tokenized_datasets['validation'])} tweets")
print(f"Données déséquilibrées : AUCUN rééquilibrage appliqué")
print("="*60 + "\n")

# Lancement de l'entraînement
trainer.train()

print("\n" + "="*60)
print(" ENTRAÎNEMENT TERMINÉ")
print("="*60)

## 3.3 Évaluer sur le test set et calculer toutes les métriques

Cette section évalue le modèle baseline sur le test set figé avec toutes les métriques obligatoires.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize

print("\n" + "="*60)
print(" ÉVALUATION SUR LE TEST SET")
print("="*60)

# Prédictions sur le test set (qui n'a jamais été modifié)
predictions_output = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions_output.predictions, axis=1)
true_labels = predictions_output.label_ids

print(f"Nombre de prédictions : {len(predictions)}")

# 1. Accuracy (pour montrer qu'elle est trompeuse)
accuracy = accuracy_score(true_labels, predictions)

# 2. F1-macro et métriques macro
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    true_labels, predictions, average='macro'
)

# 3. F1 par classe
precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
    true_labels, predictions, average=None, labels=[0, 1, 2]
)

# 4. G-mean
g_mean = geometric_mean_score(true_labels, predictions, average='macro')

# 5. AUC-PR macro
y_true_bin = label_binarize(true_labels, classes=[0, 1, 2])
y_scores = predictions_output.predictions

auc_pr_per_class = []
for i in range(3):
    precision_curve, recall_curve, _ = precision_recall_curve(y_true_bin[:, i], y_scores[:, i])
    auc_pr = auc(recall_curve, precision_curve)
    auc_pr_per_class.append(auc_pr)

auc_pr_macro = np.mean(auc_pr_per_class)

# Affichage des résultats
print("\n--- RÉSULTATS GLOBAUX ---")
print(f"  Accuracy        : {accuracy:.4f}  (MÉTRIQUE TROMPEUSE - Ne pas utiliser comme référence)")
print(f" F1-macro        : {f1_macro:.4f}  (MÉTRIQUE PRINCIPALE)")
print(f"   Precision-macro : {precision_macro:.4f}")
print(f"   Recall-macro    : {recall_macro:.4f}")
print(f"   G-mean          : {g_mean:.4f}")
print(f"   AUC-PR macro    : {auc_pr_macro:.4f}")

print("\n--- RÉSULTATS PAR CLASSE ---")
class_names = ["Negative (0)", "Neutral (1)", "Positive (2)"]
for idx, class_name in enumerate(class_names):
    print(f"\n{class_name}:")
    print(f"  F1-Score        : {f1_per_class[idx]:.4f}")
    print(f"  Precision       : {precision_per_class[idx]:.4f}")
    print(f"  Recall          : {recall_per_class[idx]:.4f}")
    print(f"  Support         : {support_per_class[idx]}")

## 3.4 Afficher le rapport de classification et la matrice de confusion

Cette section génère un rapport détaillé et visualise la matrice de confusion.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("\n" + "="*60)
print(" RAPPORT DE CLASSIFICATION COMPLET")
print("="*60 + "\n")

# Rapport de classification
target_names = ["Negative (0)", "Neutral (1)", "Positive (2)"]
report_str = classification_report(true_labels, predictions, target_names=target_names, digits=4)
print(report_str)

# Matrice de confusion
conf_matrix = confusion_matrix(true_labels, predictions)
print("\n" + "="*60)
print("MATRICE DE CONFUSION")
print("="*60)
print("\nFormat: Lignes = Vraies classes | Colonnes = Prédictions")
print("\n" + str(conf_matrix))
print()

# Visualisation
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Negative", "Neutral", "Positive"],
            yticklabels=["Negative", "Neutral", "Positive"],
            cbar_kws={'label': 'Nombre de prédictions'},
            ax=ax)
ax.set_title('Matrice de Confusion - Modèle Baseline', fontweight='bold', fontsize=14)
ax.set_ylabel('Vraie Classe', fontsize=12)
ax.set_xlabel('Classe Prédite', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix_baseline.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Matrice de confusion sauvegardée : confusion_matrix_baseline.png")

## 3.5 Sauvegarder les résultats (référence pour le tableau comparatif)

Cette section sauvegarde tous les résultats du modèle baseline pour comparaison future.

In [ ]:
import json
import pickle
import os

print("\n" + "="*60)
print(" SAUVEGARDE DES RÉSULTATS BASELINE")
print("="*60)

# Créer les répertoires de sortie
os.makedirs("./baseline_outputs", exist_ok=True)

# Structure de sauvegarde complète
baseline_results = {
    "model_name": "Baseline (données brutes déséquilibrées)",
    "description": "Modèle DziriBERT fine-tuné sur données déséquilibrées sans rééquilibrage",
    "hyperparameters": {
        "num_epochs": 5,
        "learning_rate": 2e-5,
        "batch_size": 16,
        "seed": 42,
        "optimizer": "adamw_torch",
    },
    "metrics": {
        "accuracy": float(accuracy),
        "f1_macro": float(f1_macro),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "g_mean": float(g_mean),
        "auc_pr_macro": float(auc_pr_macro),
        "auc_pr_per_class": {
            "Negative": float(auc_pr_per_class[0]),
            "Neutral": float(auc_pr_per_class[1]),
            "Positive": float(auc_pr_per_class[2]),
        }
    },
    "per_class_metrics": {
        "Negative": {
            "f1": float(f1_per_class[0]),
            "precision": float(precision_per_class[0]),
            "recall": float(recall_per_class[0]),
            "support": int(support_per_class[0]),
        },
        "Neutral": {
            "f1": float(f1_per_class[1]),
            "precision": float(precision_per_class[1]),
            "recall": float(recall_per_class[1]),
            "support": int(support_per_class[1]),
        },
        "Positive": {
            "f1": float(f1_per_class[2]),
            "precision": float(precision_per_class[2]),
            "recall": float(recall_per_class[2]),
            "support": int(support_per_class[2]),
        },
    },
    "confusion_matrix": conf_matrix.tolist(),
    "classification_report": report_str,
}

# Sauvegarde JSON
json_path = './baseline_outputs/baseline_results.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(baseline_results, f, indent=4, ensure_ascii=False)
print(f" Résultats JSON : {json_path}")

# Sauvegarde pickle
pkl_path = './baseline_outputs/baseline_results.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(baseline_results, f)
print(f" Résultats pickle : {pkl_path}")

# Sauvegarde du modèle
model_path = "./baseline_outputs/baseline_model"
trainer.save_model(model_path)
print(f" Modèle sauvegardé : {model_path}")

# Sauvegarde du tokenizer
tokenizer.save_pretrained(model_path)
print(f" Tokenizer sauvegardé")

print("\n" + "="*60)
print(" RÉSUMÉ BASELINE - RÉFÉRENCE POUR LE TABLEAU COMPARATIF")
print("="*60)
print(f"\nF1-macro BASELINE  : {f1_macro:.4f}")
print(f"G-mean BASELINE    : {g_mean:.4f}")
print(f"AUC-PR BASELINE    : {auc_pr_macro:.4f}")
print("\nCes scores servent de référence pour évaluer l'amélioration")
print("des stratégies de rééquilibrage (Étapes 4, 5, 6).")
print("="*60)

## Étape 4 — Stratégie 1 : Modification de la Fonction de Perte

Principe fondamental de cette stratégie : On ne modifie pas les données. Le corpus reste déséquilibré tel quel. On change uniquement la façon dont le modèle apprend à partir de ses erreurs. C'est une approche au niveau de l'algorithme, pas au niveau des données.

Par défaut, DziriBERT utilise une CrossEntropy standard qui traite toutes les classes de la même façon. Si une classe est majoritaire, le modèle apprend à la prédire très vite. La modification de la fonction de perte corrige ce biais en signalant au modèle : une erreur sur la classe minoritaire coûte plus cher.

### 4.1 Variante A — Class Weighting

C'est la méthode la plus simple et la plus directe. On calcule un poids inversement proportionnel à la fréquence de chaque classe, puis on l'injecte dans la CrossEntropy.

Formule : poids = total_exemples / (nb_classes × nb_exemples_de_la_classe)

On garde exactement les mêmes hyperparamètres que la Baseline.

In [ ]:
# 4.1 Variante A — Class Weighting

from transformers import EarlyStoppingCallback, AutoModelForSequenceClassification
import torch.nn as nn

if 'label' in tokenized_datasets['train'].column_names and 'labels' not in tokenized_datasets['train'].column_names:
    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
    print(" Colonne renommée 'label' → 'labels'")
else:
    print(f"Colonnes actuelles : {tokenized_datasets['train'].column_names}")

# Calcul des poids de classe sur le train set
counts = train_df['label'].value_counts().sort_index()  # [Negative, Neutral, Positive]
total = len(train_df)
num_classes = 3
weights = torch.tensor([total / (num_classes * counts[i]) for i in range(3)], dtype=torch.float).to(device)

print("Nombre d'exemples par classe dans le train set:")
print(f"Negative (0): {counts[0]} | Neutral (1): {counts[1]} | Positive (2): {counts[2]}")
print(f"Poids calculés: {weights}")

# Modèle personnalisé pour utiliser CrossEntropy avec poids
class WeightedModel(nn.Module):
    def __init__(self, model_name, weights, device):
        super().__init__()
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
        self.loss_fn = nn.CrossEntropyLoss(weight=weights.to(device))
        self.device = device

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        loss = None
        if labels is not None:
            loss = self.loss_fn(outputs.logits, labels)

        # Retourner un objet SequenceClassifierOutput compatible avec le Trainer
        from transformers import modeling_outputs
        return modeling_outputs.SequenceClassifierOutput(
            loss=loss,
            logits=outputs.logits,
            hidden_states=outputs.hidden_states if hasattr(outputs, 'hidden_states') else None,
            attentions=outputs.attentions if hasattr(outputs, 'attentions') else None,
        )

# Initialisation du modèle avec poids
set_seed(42)  # Reproductibilité
model_weighted = WeightedModel(model_checkpoint, weights, device)
model_weighted = model_weighted.to(device)

# Configuration du Trainer (mêmes hyperparamètres)
trainer_weighted = Trainer(
    model=model_weighted,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\n" + "="*60)
print(" ENTRAÎNEMENT - CLASS WEIGHTING")
print("="*60)
trainer_weighted.train()

print("\n" + "="*60)
print(" ÉVALUATION SUR LE TEST SET - CLASS WEIGHTING")
print("="*60)

# Prédictions
predictions_output = trainer_weighted.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions_output.predictions, axis=1)
true_labels = predictions_output.label_ids

# Métriques
accuracy = accuracy_score(true_labels, predictions)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(true_labels, predictions, average='macro')
precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(true_labels, predictions, average=None, labels=[0, 1, 2])
g_mean = geometric_mean_score(true_labels, predictions, average='macro')
y_true_bin = label_binarize(true_labels, classes=[0, 1, 2])
y_scores = predictions_output.predictions
auc_pr_per_class = []
for i in range(3):
    precision_curve, recall_curve, _ = precision_recall_curve(y_true_bin[:, i], y_scores[:, i])
    auc_pr = auc(recall_curve, precision_curve)
    auc_pr_per_class.append(auc_pr)
auc_pr_macro = np.mean(auc_pr_per_class)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-macro: {f1_macro:.4f}")
print(f"G-mean: {g_mean:.4f}")
print(f"AUC-PR macro: {auc_pr_macro:.4f}")

# Rapport de classification
report_str = classification_report(true_labels, predictions, target_names=["Negative", "Neutral", "Positive"], digits=4)
print("\nRapport de classification:")
print(report_str)

# Sauvegarde des résultats
os.makedirs("./strategy1_outputs", exist_ok=True)
weighted_results = {
    "model_name": "Class Weighting",
    "weights": weights.tolist(),
    "metrics": {
        "accuracy": float(accuracy),
        "f1_macro": float(f1_macro),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "g_mean": float(g_mean),
        "auc_pr_macro": float(auc_pr_macro),
        "auc_pr_per_class": {
            "Negative": float(auc_pr_per_class[0]),
            "Neutral": float(auc_pr_per_class[1]),
            "Positive": float(auc_pr_per_class[2]),
        }
    },
    "per_class_metrics": {
        "Negative": {"f1": float(f1_per_class[0]), "precision": float(precision_per_class[0]), "recall": float(recall_per_class[0]), "support": int(support_per_class[0])},
        "Neutral": {"f1": float(f1_per_class[1]), "precision": float(precision_per_class[1]), "recall": float(recall_per_class[1]), "support": int(support_per_class[1])},
        "Positive": {"f1": float(f1_per_class[2]), "precision": float(precision_per_class[2]), "recall": float(recall_per_class[2]), "support": int(support_per_class[2])},
    },
    "confusion_matrix": confusion_matrix(true_labels, predictions).tolist(),
    "classification_report": report_str,
}

with open('./strategy1_outputs/class_weighting_results.json', 'w', encoding='utf-8') as f:
    json.dump(weighted_results, f, indent=4, ensure_ascii=False)
with open('./strategy1_outputs/class_weighting_results.pkl', 'wb') as f:
    pickle.dump(weighted_results, f)

print(" Résultats Class Weighting sauvegardés")

### 4.2 Variante B — Focal Loss

La Focal Loss réduit la contribution des exemples faciles et se concentre sur les difficiles. Paramètre gamma contrôle l'effet.

On teste gamma=1 et gamma=2.

In [ ]:
# 4.2 Variante B — Focal Loss

# Imports essentiels pour cette cellule
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
from transformers import modeling_outputs, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
from imblearn.metrics import geometric_mean_score
import json
import pickle
import numpy as np

# Définir set_seed localement (au cas où la cellule serait réexécutée indépendamment)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Définir les variables globales si elles ne sont pas disponibles
model_checkpoint = "alger-ia/dziribert"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Définir training_args localement si pas disponible
try:
    training_args
except NameError:
    from transformers import TrainingArguments
    training_args = TrainingArguments(
        output_dir="./results_focal",
        num_train_epochs=5,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        seed=42,
        optim="adamw_torch",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        weight_decay=0.01,
        report_to="none",
    )

# Définir compute_metrics localement si pas disponible
try:
    compute_metrics
except NameError:
    from sklearn.metrics import precision_recall_fscore_support
    from imblearn.metrics import geometric_mean_score

    def compute_metrics(eval_pred):
        """Calcule les métriques obligatoires de la Section 5.3"""
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=1)

        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
            labels, predictions, average='macro'
        )

        _, _, f1_per_class, _ = precision_recall_fscore_support(
            labels, predictions, average=None, labels=[0, 1, 2]
        )

        g_mean = geometric_mean_score(labels, predictions, average='macro')

        return {
            "f1_macro": f1_macro,
            "precision_macro": precision_macro,
            "recall_macro": recall_macro,
            "g_mean": g_mean,
            "f1_negative": f1_per_class[0],
            "f1_neutral": f1_per_class[1],
            "f1_positive": f1_per_class[2],
        }

# Note: This cell expects tokenized_datasets, train_df, and training_args to be defined
# from earlier cells. Make sure to run all preceding cells first.

# Classe Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, gamma=2, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# Modèle personnalisé pour Focal Loss
class FocalModel(nn.Module):
    def __init__(self, model_name, gamma, device):
        super().__init__()
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
        self.loss_fn = FocalLoss(gamma=gamma)
        self.device = device

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        loss = None
        if labels is not None:
            loss = self.loss_fn(outputs.logits, labels)

        # Retourner un objet SequenceClassifierOutput compatible avec le Trainer
        return modeling_outputs.SequenceClassifierOutput(
            loss=loss,
            logits=outputs.logits,
            hidden_states=outputs.hidden_states if hasattr(outputs, 'hidden_states') else None,
            attentions=outputs.attentions if hasattr(outputs, 'attentions') else None,
        )

# Fonction pour entraîner et évaluer Focal Loss
def train_and_evaluate_focal(gamma):
    print(f"\n" + "="*60)
    print(f" ENTRAÎNEMENT - FOCAL LOSS (gamma={gamma})")
    print("="*60)

    set_seed(42)
    model_focal = FocalModel(model_checkpoint, gamma, device)
    model_focal = model_focal.to(device)

    trainer_focal = Trainer(
        model=model_focal,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer_focal.train()

    print(f"\n ÉVALUATION SUR LE TEST SET - FOCAL LOSS (gamma={gamma})")

    predictions_output = trainer_focal.predict(tokenized_datasets["test"])
    predictions = np.argmax(predictions_output.predictions, axis=1)
    true_labels = predictions_output.label_ids

    accuracy = accuracy_score(true_labels, predictions)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(true_labels, predictions, average='macro')
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(true_labels, predictions, average=None, labels=[0, 1, 2])
    g_mean = geometric_mean_score(true_labels, predictions, average='macro')
    y_true_bin = label_binarize(true_labels, classes=[0, 1, 2])
    y_scores = predictions_output.predictions
    auc_pr_per_class = []
    for i in range(3):
        precision_curve, recall_curve, _ = precision_recall_curve(y_true_bin[:, i], y_scores[:, i])
        auc_pr = auc(recall_curve, precision_curve)
        auc_pr_per_class.append(auc_pr)
    auc_pr_macro = np.mean(auc_pr_per_class)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-macro: {f1_macro:.4f}")
    print(f"G-mean: {g_mean:.4f}")
    print(f"AUC-PR macro: {auc_pr_macro:.4f}")

    report_str = classification_report(true_labels, predictions, target_names=["Negative", "Neutral", "Positive"], digits=4)
    print("\nRapport de classification:")
    print(report_str)

    # Sauvegarde
    focal_results = {
        "model_name": f"Focal Loss (gamma={gamma})",
        "gamma": gamma,
        "metrics": {
            "accuracy": float(accuracy),
            "f1_macro": float(f1_macro),
            "precision_macro": float(precision_macro),
            "recall_macro": float(recall_macro),
            "g_mean": float(g_mean),
            "auc_pr_macro": float(auc_pr_macro),
            "auc_pr_per_class": {
                "Negative": float(auc_pr_per_class[0]),
                "Neutral": float(auc_pr_per_class[1]),
                "Positive": float(auc_pr_per_class[2]),
            }
        },
        "per_class_metrics": {
            "Negative": {"f1": float(f1_per_class[0]), "precision": float(precision_per_class[0]), "recall": float(recall_per_class[0]), "support": int(support_per_class[0])},
            "Neutral": {"f1": float(f1_per_class[1]), "precision": float(precision_per_class[1]), "recall": float(recall_per_class[1]), "support": int(support_per_class[1])},
            "Positive": {"f1": float(f1_per_class[2]), "precision": float(precision_per_class[2]), "recall": float(recall_per_class[2]), "support": int(support_per_class[2])},
        },
        "confusion_matrix": confusion_matrix(true_labels, predictions).tolist(),
        "classification_report": report_str,
    }

    with open(f'./strategy1_outputs/focal_loss_gamma_{gamma}_results.json', 'w', encoding='utf-8') as f:
        json.dump(focal_results, f, indent=4, ensure_ascii=False)
    with open(f'./strategy1_outputs/focal_loss_gamma_{gamma}_results.pkl', 'wb') as f:
        pickle.dump(focal_results, f)

    print(f" Résultats Focal Loss (gamma={gamma}) sauvegardés")

    return focal_results

# Appel de train_and_evaluate_focal pour les deux gammas
print("\n" + "="*60)
print(" ENTRAÎNEMENT FOCAL LOSS AVEC DIFFÉRENTS GAMMAS")
print("="*60)

focal_gamma1_results = train_and_evaluate_focal(gamma=1)
focal_gamma2_results = train_and_evaluate_focal(gamma=2)

print("\n" + "="*60)
print(" RÉSUMÉ FOCAL LOSS")
print("="*60)
print(f"Focal Loss (gamma=1) F1-macro: {focal_gamma1_results['metrics']['f1_macro']:.4f}")
print(f"Focal Loss (gamma=2) F1-macro: {focal_gamma2_results['metrics']['f1_macro']:.4f}")


### 4.3 Variante C — Combinaison Class Weighting + Focal Loss

On combine les poids de classe et la Focal Loss pour amplifier l'effet. Ici, on utilise gamma=1 avec les poids calculés.

In [ ]:
# 4.3 Variante C — Combinaison Class Weighting + Focal Loss

from transformers import EarlyStoppingCallback, AutoModelForSequenceClassification
import torch.nn as nn

# Modifier FocalLoss pour accepter les poids de classe (alpha)
class FocalLossWeighted(nn.Module):
    def __init__(self, alpha=None, gamma=1, reduction='mean'):
        super().__init__()
        self.alpha = alpha  # tensor des poids
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# Modèle pour la combinaison
class CombinedModel(nn.Module):
    def __init__(self, model_name, weights, gamma=1, device=None):
        super().__init__()
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
        self.loss_fn = FocalLossWeighted(alpha=weights.to(device), gamma=gamma)
        self.device = device

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        loss = None
        if labels is not None:
            loss = self.loss_fn(outputs.logits, labels)

        # Retourner un objet SequenceClassifierOutput compatible avec le Trainer
        from transformers import modeling_outputs
        return modeling_outputs.SequenceClassifierOutput(
            loss=loss,
            logits=outputs.logits,
            hidden_states=outputs.hidden_states if hasattr(outputs, 'hidden_states') else None,
            attentions=outputs.attentions if hasattr(outputs, 'attentions') else None,
        )

print("\n" + "="*60)
print(" ENTRAÎNEMENT - COMBINAISON CLASS WEIGHTING + FOCAL LOSS (gamma=1)")
print("="*60)

set_seed(42)
model_combined = CombinedModel(model_checkpoint, weights, gamma=1, device=device)
model_combined = model_combined.to(device)

trainer_combined = Trainer(
    model=model_combined,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer_combined.train()

print("\n ÉVALUATION SUR LE TEST SET - COMBINAISON")

predictions_output = trainer_combined.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions_output.predictions, axis=1)
true_labels = predictions_output.label_ids

accuracy = accuracy_score(true_labels, predictions)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(true_labels, predictions, average='macro')
precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(true_labels, predictions, average=None, labels=[0, 1, 2])
g_mean = geometric_mean_score(true_labels, predictions, average='macro')
y_true_bin = label_binarize(true_labels, classes=[0, 1, 2])
y_scores = predictions_output.predictions
auc_pr_per_class = []
for i in range(3):
    precision_curve, recall_curve, _ = precision_recall_curve(y_true_bin[:, i], y_scores[:, i])
    auc_pr = auc(recall_curve, precision_curve)
    auc_pr_per_class.append(auc_pr)
auc_pr_macro = np.mean(auc_pr_per_class)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-macro: {f1_macro:.4f}")
print(f"G-mean: {g_mean:.4f}")
print(f"AUC-PR macro: {auc_pr_macro:.4f}")

report_str = classification_report(true_labels, predictions, target_names=["Negative", "Neutral", "Positive"], digits=4)
print("\nRapport de classification:")
print(report_str)

# Sauvegarde
combined_results = {
    "model_name": "Class Weighting + Focal Loss (gamma=1)",
    "weights": weights.tolist(),
    "gamma": 1,
    "metrics": {
        "accuracy": float(accuracy),
        "f1_macro": float(f1_macro),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "g_mean": float(g_mean),
        "auc_pr_macro": float(auc_pr_macro),
        "auc_pr_per_class": {
            "Negative": float(auc_pr_per_class[0]),
            "Neutral": float(auc_pr_per_class[1]),
            "Positive": float(auc_pr_per_class[2]),
        }
    },
    "per_class_metrics": {
        "Negative": {"f1": float(f1_per_class[0]), "precision": float(precision_per_class[0]), "recall": float(recall_per_class[0]), "support": int(support_per_class[0])},
        "Neutral": {"f1": float(f1_per_class[1]), "precision": float(precision_per_class[1]), "recall": float(recall_per_class[1]), "support": int(support_per_class[1])},
        "Positive": {"f1": float(f1_per_class[2]), "precision": float(precision_per_class[2]), "recall": float(recall_per_class[2]), "support": int(support_per_class[2])},
    },
    "confusion_matrix": confusion_matrix(true_labels, predictions).tolist(),
    "classification_report": report_str,
}

with open('./strategy1_outputs/combined_weighting_focal_results.json', 'w', encoding='utf-8') as f:
    json.dump(combined_results, f, indent=4, ensure_ascii=False)
with open('./strategy1_outputs/combined_weighting_focal_results.pkl', 'wb') as f:
    pickle.dump(combined_results, f)

print(" Résultats Combinaison sauvegardés")

# Appel de train_and_evaluate_focal pour les deux gammas
print("\n" + "="*60)
print(" ENTRAÎNEMENT FOCAL LOSS AVEC DIFFÉRENTS GAMMAS")
print("="*60)

train_and_evaluate_focal(gamma=1)
train_and_evaluate_focal(gamma=2)

print("\n" + "="*60)
print("RÉSUMÉ ÉTAPE 4 — STRATÉGIE 1")
print("="*60)
print("Toutes les variantes ont été entraînées et évaluées.")
print("Les résultats sont sauvegardés dans ./strategy1_outputs/")
print("Comparez avec la baseline dans ./baseline_outputs/")

ETAPE 5 — Stratégie 2 : Rééquilibrage dans l'Espace des Embeddings

5.1 Le vecteur [CLS] de DziriBERT

In [ ]:
# 5.1 Extraction des vecteurs [CLS] de DziriBERT

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
import os
import pickle
from tqdm import tqdm

print("\n" + "="*60)
print(" EXTRACTION DES EMBEDDINGS [CLS] - ÉTAPE 5.1")
print("="*60)

# Configuration GPU/CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# 1. Charger le modèle en mode extraction de features (sans tête de classification)
print("\nChargement de DziriBERT en mode feature extraction...")
model_checkpoint = "alger-ia/dziribert"

# Charger le modèle de base (AutoModel) sans la tête de classification
feature_model = AutoModel.from_pretrained(model_checkpoint)
feature_model = feature_model.to(device)
feature_model.eval()  # Mode évaluation

print(" Modèle chargé avec succès en mode feature extraction")

# 2. Charger le tokenizer (vérifier d'abord s'il existe dans le contexte)
print("\nChargement du tokenizer...")
try:
    # Essayer d'utiliser le tokenizer global s'il existe
    if 'tokenizer' in globals() and tokenizer is not None:
        print(" Tokenizer déjà chargé (depuis contexte global)")
    else:
        raise NameError
except NameError:
    # Sinon, le charger depuis Hugging Face
    print("Tokenizer non trouvé, chargement depuis Hugging Face...")
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    print(" Tokenizer chargé depuis Hugging Face")

# 3. Fonction pour extraire l'embedding [CLS] d'un tweet
def extract_cls_embedding(text, tokenizer, model, device, max_length=128):
    """
    Extrait le vecteur [CLS] pour un tweet donné.

    Args:
        text (str): Le texte du tweet
        tokenizer: Tokenizer de DziriBERT
        model: Modèle DziriBERT chargé
        device: GPU ou CPU
        max_length (int): Longueur maximale de la séquence

    Returns:
        numpy array de forme (768,): Le vecteur [CLS]
    """
    # Tokeniser le texte
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_length
    ).to(device)

    # Extraire les embeddings
    with torch.no_grad():
        outputs = model(**inputs)
        # outputs.last_hidden_state a la forme (batch_size, seq_length, 768)
        # Le premier token est [CLS] (indice 0)
        cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()

    return cls_embedding.squeeze()  # Supprimer la dimension batch

# 4. Vérifier que train_df existe et charger si nécessaire
print("\nVérification du train_df...")
try:
    if 'train_df' not in globals() or train_df is None:
        raise NameError("train_df non trouvé")
    print(f" train_df trouvé ({len(train_df)} exemples)")
except NameError:
    print(" ERREUR: train_df n'est pas défini.")
    print("  Assurez-vous d'avoir exécuté toutes les cellules précédentes (surtout la cellule 2.4 de split train/val/test)")
    raise

# 5. Boucler sur le train set et extraire les embeddings
print("\n--- EXTRACTION DES EMBEDDINGS [CLS] ---")
print(f"Traitement de {len(train_df)} tweets du train set...")

embeddings = []
labels = []

with torch.no_grad():
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Extraction des embeddings"):
        text = row['Post']
        label = row['label']  # label est déjà un entier après le mapping (0, 1, 2)

        # Extraire l'embedding
        cls_vec = extract_cls_embedding(text, tokenizer, feature_model, device)
        embeddings.append(cls_vec)
        labels.append(label)

# 6. Convertir les listes en arrays numpy
embeddings = np.array(embeddings)
labels = np.array(labels)

print(f"\n Extraction terminée !")
print(f"Matrice d'embeddings shape: {embeddings.shape}")
print(f"Vecteur labels shape: {labels.shape}")

# 7. Vérification : afficher les statistiques
print("\n--- VÉRIFICATION DES EMBEDDINGS ---")
print(f"Valeur min: {embeddings.min():.6f}")
print(f"Valeur max: {embeddings.max():.6f}")
print(f"Valeur moyenne: {embeddings.mean():.6f}")
print(f"Écart-type: {embeddings.std():.6f}")

# Distribution des classes
unique_labels, counts = np.unique(labels, return_counts=True)
print(f"\nDistribution des classes dans le train set (embeddings):")
for label, count in zip(unique_labels, counts):
    class_name = {0: "Negative", 1: "Neutral", 2: "Positive"}[label]
    print(f"  {class_name}: {count} exemples")

# 8. Sauvegarder les embeddings et labels
print("\n--- SAUVEGARDE ---")
os.makedirs("./embeddings_outputs", exist_ok=True)

# Sauvegarder sous format NumPy (.npz)
np.savez_compressed(
    "./embeddings_outputs/train_embeddings.npz",
    embeddings=embeddings,
    labels=labels
)
print(" Embeddings et labels sauvegardés : ./embeddings_outputs/train_embeddings.npz")

# Sauvegarder aussi sous format pickle pour la flexibilité
with open("./embeddings_outputs/train_embeddings.pkl", "wb") as f:
    pickle.dump({"embeddings": embeddings, "labels": labels}, f)
print(" Embeddings et labels sauvegardés : ./embeddings_outputs/train_embeddings.pkl")

# 9. Afficher une analyse des embeddings
print("\n" + "="*60)
print(" ANALYSE DES EMBEDDINGS EXTRAITS")
print("="*60)

# Vérifier que les embeddings sont bien vectorisés
print(f"\nDimension des embeddings : {embeddings.shape[1]}")  # Doit être 768

# Afficher quelques statistiques par classe
for class_id in [0, 1, 2]:
    class_embeddings = embeddings[labels == class_id]
    class_name = {0: "Negative", 1: "Neutral", 2: "Positive"}[class_id]
    print(f"\n{class_name} (classe {class_id}):")
    print(f"  Nombre d'exemples: {len(class_embeddings)}")
    print(f"  Norme L2 moyenne: {np.linalg.norm(class_embeddings, axis=1).mean():.4f}")
    print(f"  Écart-type de la norme: {np.linalg.norm(class_embeddings, axis=1).std():.4f}")

print("\n" + "="*60)
print(" ÉTAPE 5.1 TERMINÉE")
print("="*60)
print("\nLes embeddings sont prêts pour:")
print("  → SMOTE (Section 5.2)")
print("  → ADASYN (Section 5.3)")
print("\n RAPPEL : N'extraire les embeddings QUE du train set")
print("   Le val set et le test set restent intacts pour l'évaluation")


**5.2** Variante A — SMOTE (Synthetic Minority Over-sampling Technique)

In [ ]:
# ============================================================================
# 5.2 Variante A — SMOTE AMÉLIORÉ (Pour de meilleures performances)
# ============================================================================
# Objectif : Augmenter le F1-macro de 0.56 vers 0.62-0.65
# Corrections appliquées :
#   1. Équilibrage PARTIEL seulement (éviter de sur-générer pour Neutral)
#   2. MLP plus profond et mieux régularisé
#   3. Augmentation du k_neighbors de SMOTE (plus de diversité)
#   4. Test de plusieurs stratégies d'échantillonnage optimisées
# ============================================================================

import numpy as np
import pickle
import json
import os
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
    precision_recall_curve, auc
)
from sklearn.preprocessing import label_binarize
from imblearn.metrics import geometric_mean_score
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import AutoTokenizer, AutoModel

print("\n" + "="*80)
print(" ÉTAPE 5.2 — SMOTE AMÉLIORÉ (Version Optimisée)")
print("="*80)

# Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# 1. Charger les embeddings du train set
print("\n--- CHARGEMENT DES EMBEDDINGS ---")
try:
    embeddings_data = np.load("./embeddings_outputs/train_embeddings.npz")
    train_embeddings = embeddings_data['embeddings']
    train_labels = embeddings_data['labels']
    print(f"✅ Embeddings chargés : {train_embeddings.shape}")
    print(f"✅ Labels chargés : {train_labels.shape}")
except FileNotFoundError:
    print("❌ ERREUR: Les embeddings n'ont pas été trouvés.")
    raise

# 2. Distribution des classes
unique_labels, counts = np.unique(train_labels, return_counts=True)
class_names = {0: "Negative", 1: "Neutral", 2: "Positive"}
print("\n--- DISTRIBUTION DES CLASSES AVANT SMOTE ---")
for label, count in zip(unique_labels, counts):
    print(f"  {class_names[label]}: {count} exemples")

majority_count = counts.max()
minority_count = counts.min()
print(f"\n📊 Ratio de déséquilibre: {majority_count/minority_count:.2f}")

# ============================================================================
# FONCTION AMÉLIORÉE POUR SMOTE AVEC MLP OPTIMISÉ
# ============================================================================

def apply_smote_improved(embeddings, labels, target_ratio, name="Variante"):
    """
    Applique SMOTE avec un ratio cible pour la classe minoritaire
    Au lieu d'équilibrer complètement, on cible un ratio spécifique

    Args:
        embeddings: Matrice d'embeddings
        labels: Vecteur de labels
        target_ratio: Ratio cible pour la classe minoritaire (ex: 0.6 = 60% de la majorité)
        name: Nom de la variante
    """
    print(f"\n--- APPLICATION SMOTE ({name}) ---")
    print(f"  🎯 Ratio cible: {target_ratio} ({target_ratio*100}% de la classe majoritaire)")

    # Calculer le nombre cible pour les classes minoritaires
    current_counts = np.unique(labels, return_counts=True)[1]

    # Trouver la classe majoritaire (Positive avec 1608)
    majority_class = 2  # Positive
    majority_size = current_counts[2]  # 1608

    # Pour les classes minoritaires (Negative et Neutral)
    # On les sur-échantillonne pour atteindre target_ratio * majority_size
    target_neutral = int(majority_size * target_ratio)
    target_negative = int(majority_size * target_ratio)

    # S'assurer qu'on ne réduit pas les classes (pas de sous-échantillonnage)
    target_neutral = max(target_neutral, current_counts[1])
    target_negative = max(target_negative, current_counts[0])

    # Construire le dictionnaire de stratégie
    sampling_strategy = {
        0: target_negative,   # Negative
        1: target_neutral,    # Neutral
        2: majority_size      # Positive reste inchangée
    }

    print(f"  📊 Stratégie d'échantillonnage:")
    print(f"     • Negative: {current_counts[0]} → {target_negative} (+{((target_negative-current_counts[0])/current_counts[0])*100:.1f}%)")
    print(f"     • Neutral:  {current_counts[1]} → {target_neutral} (+{((target_neutral-current_counts[1])/current_counts[1])*100:.1f}%)")
    print(f"     • Positive: {current_counts[2]} → {majority_size} (+0%)")

    # SMOTE avec k_neighbors plus faible pour éviter la dispersion
    smote = SMOTE(
        sampling_strategy=sampling_strategy,
        random_state=42,
        k_neighbors=3  # Réduit de 5 à 3 pour une interpolation plus locale
    )

    new_embeddings, new_labels = smote.fit_resample(embeddings, labels)

    print(f"\n  📈 Taille finale: {len(embeddings)} → {len(new_embeddings)} exemples")

    return new_embeddings, new_labels


def train_mlp_improved(X_train, y_train, X_test, y_test, hidden_layers, alpha, learning_rate_init):
    """
    Entraîne un MLP optimisé avec early stopping et régularisation
    """
    mlp = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation='relu',
        solver='adam',
        alpha=alpha,  # Régularisation L2
        learning_rate='adaptive',
        learning_rate_init=learning_rate_init,
        batch_size=32,
        max_iter=300,  # Augmenté
        random_state=42,
        early_stopping=True,
        validation_fraction=0.15,  # Augmenté pour meilleure validation
        verbose=False
    )

    mlp.fit(X_train, y_train)
    predictions = mlp.predict(X_test)

    return mlp, predictions


# ============================================================================
# TEST DE DIFFÉRENTES CONFIGURATIONS OPTIMISÉES
# ============================================================================

print("\n" + "="*80)
print(" TEST DE CONFIGURATIONS OPTIMISÉES")
print("="*80)

# Charger les embeddings du test set
try:
    test_embeddings_data = np.load("./embeddings_outputs/test_embeddings.npz")
    test_embeddings = test_embeddings_data['embeddings']
    test_labels = test_embeddings_data['labels']
    print("✅ Embeddings test set chargés")
except FileNotFoundError:
    print("⚠️  Extraction des embeddings du test set...")
    model_checkpoint = "alger-ia/dziribert"
    feature_model = AutoModel.from_pretrained(model_checkpoint).to(device)
    feature_model.eval()
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    test_embeddings = []
    test_labels = []
    for idx, row in test_df.iterrows():
        text = row['Post']
        label = row['label']
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                          padding="max_length", max_length=128).to(device)
        with torch.no_grad():
            outputs = feature_model(**inputs)
            cls_vec = outputs.last_hidden_state[:, 0, :].cpu().numpy().squeeze()
        test_embeddings.append(cls_vec)
        test_labels.append(label)

    test_embeddings = np.array(test_embeddings)
    test_labels = np.array(test_labels)
    np.savez_compressed("./embeddings_outputs/test_embeddings.npz",
                       embeddings=test_embeddings, labels=test_labels)

# ============================================================================
# CONFIGURATIONS À TESTER (optimisées pour maximiser F1-macro)
# ============================================================================

configurations = [
    # (target_ratio, hidden_layers, alpha, lr_init, name)
    (0.35, (1024, 512, 256, 128), 0.005, 0.0005, "SMOTE_Ratio35_DeepNet"),    # ~35% de la majorité
    (0.40, (1024, 512, 256, 128), 0.005, 0.0005, "SMOTE_Ratio40_DeepNet"),    # ~40% - OPTIMAL PRÉSUME
    (0.45, (1024, 512, 256, 128), 0.005, 0.0005, "SMOTE_Ratio45_DeepNet"),
    (0.50, (1024, 512, 256), 0.01, 0.001, "SMOTE_Ratio50_MediumNet"),         # Version original améliorée
    (0.35, (512, 256, 128, 64), 0.01, 0.001, "SMOTE_Ratio35_RandomForest"),   # Avec RandomForest
]

# Pour stocker les résultats
improved_results = {}
os.makedirs("./strategy2_outputs_improved", exist_ok=True)

for target_ratio, hidden_layers, alpha, lr_init, config_name in configurations:
    print(f"\n{'='*80}")
    print(f" 📊 CONFIGURATION: {config_name}")
    print(f"{'='*80}")

    # Appliquer SMOTE avec ratio cible
    smote_embeddings, smote_labels = apply_smote_improved(
        train_embeddings, train_labels,
        target_ratio=target_ratio,
        name=config_name
    )

    # Normalisation
    scaler = StandardScaler()
    smote_embeddings_scaled = scaler.fit_transform(smote_embeddings)
    test_embeddings_scaled = scaler.transform(test_embeddings)

    # Choix du classifieur
    if "RandomForest" in config_name:
        print(f"  🌲 Utilisation de RandomForest (plus robuste que MLP)")
        classifier = RandomForestClassifier(
            n_estimators=200,
            max_depth=20,
            min_samples_split=5,
            random_state=42,
            n_jobs=-1
        )
        classifier.fit(smote_embeddings_scaled, smote_labels)
        predictions = classifier.predict(test_embeddings_scaled)
        n_iter = "N/A"

    else:
        print(f"  🤖 MLP avec architecture {hidden_layers}")
        classifier, predictions = train_mlp_improved(
            smote_embeddings_scaled, smote_labels,
            test_embeddings_scaled, test_labels,
            hidden_layers=hidden_layers,
            alpha=alpha,
            learning_rate_init=lr_init
        )
        n_iter = getattr(classifier, 'n_iter_', 'N/A')

    # Calcul des métriques
    accuracy = accuracy_score(test_labels, predictions)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        test_labels, predictions, average='macro'
    )
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
        test_labels, predictions, average=None, labels=[0, 1, 2]
    )
    g_mean = geometric_mean_score(test_labels, predictions, average='macro')

    # AUC-PR
    y_true_bin = label_binarize(test_labels, classes=[0, 1, 2])
    if "RandomForest" in config_name:
        y_scores = classifier.predict_proba(test_embeddings_scaled)
    else:
        y_scores = classifier.predict_proba(test_embeddings_scaled)

    auc_pr_per_class = []
    for i in range(3):
        precision_curve, recall_curve, _ = precision_recall_curve(y_true_bin[:, i], y_scores[:, i])
        auc_pr = auc(recall_curve, precision_curve)
        auc_pr_per_class.append(auc_pr)
    auc_pr_macro = np.mean(auc_pr_per_class)

    # Affichage des résultats
    print(f"\n  📈 RÉSULTATS:")
    print(f"     ✅ F1-macro: {f1_macro:.4f} {'🎯' if f1_macro >= 0.62 else '⚠️'}")
    print(f"     📊 Accuracy : {accuracy:.4f}")
    print(f"     🎯 G-mean   : {g_mean:.4f}")
    print(f"     📉 AUC-PR   : {auc_pr_macro:.4f}")
    print(f"\n     Par classe:")
    print(f"        Negative: F1={f1_per_class[0]:.4f} (P={precision_per_class[0]:.4f}, R={recall_per_class[0]:.4f})")
    print(f"        Neutral:  F1={f1_per_class[1]:.4f} (P={precision_per_class[1]:.4f}, R={recall_per_class[1]:.4f})")
    print(f"        Positive: F1={f1_per_class[2]:.4f} (P={precision_per_class[2]:.4f}, R={recall_per_class[2]:.4f})")

    # Matrice de confusion
    conf_matrix = confusion_matrix(test_labels, predictions)
    print(f"\n  📊 Matrice de confusion:")
    print(f"     [[{conf_matrix[0,0]:3d} {conf_matrix[0,1]:3d} {conf_matrix[0,2]:3d}]")
    print(f"      [{conf_matrix[1,0]:3d} {conf_matrix[1,1]:3d} {conf_matrix[1,2]:3d}]")
    print(f"      [{conf_matrix[2,0]:3d} {conf_matrix[2,1]:3d} {conf_matrix[2,2]:3d}]]")

    # Sauvegarde
    improved_results[config_name] = {
        "target_ratio": target_ratio,
        "hidden_layers": hidden_layers,
        "alpha": alpha,
        "learning_rate_init": lr_init,
        "n_iterations": n_iter if n_iter != 'N/A' else None,
        "metrics": {
            "accuracy": float(accuracy),
            "f1_macro": float(f1_macro),
            "precision_macro": float(precision_macro),
            "recall_macro": float(recall_macro),
            "g_mean": float(g_mean),
            "auc_pr_macro": float(auc_pr_macro),
        },
        "per_class_f1": {
            "Negative": float(f1_per_class[0]),
            "Neutral": float(f1_per_class[1]),
            "Positive": float(f1_per_class[2]),
        },
        "confusion_matrix": conf_matrix.tolist(),
    }

    # ========================================================================
    # VISUALISATION AVEC AFFICHAGE À L'ÉCRAN (CORRECTION APPLIQUÉE)
    # ========================================================================
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Negative", "Neutral", "Positive"],
                yticklabels=["Negative", "Neutral", "Positive"],
                ax=ax)
    ax.set_title(f'Matrice de Confusion - {config_name}\nF1-macro = {f1_macro:.4f}',
                 fontweight='bold', fontsize=12)
    ax.set_ylabel('Vraie Classe')
    ax.set_xlabel('Classe Prédite')
    plt.tight_layout()
    plt.savefig(f'./strategy2_outputs_improved/confusion_{config_name}.png',
                dpi=300, bbox_inches='tight')
    plt.show()  # ✅ AFFICHE LA MATRICE DE CONFUSION À L'ÉCRAN
    print(f"  💾 Graphique sauvegardé: ./strategy2_outputs_improved/confusion_{config_name}.png")
    print(f"  🖥️  Graphique affiché à l'écran")

# ============================================================================
# RÉSUMÉ ET MEILLEURE CONFIGURATION
# ============================================================================

print("\n" + "="*80)
print(" 📊 RÉSUMÉ DES RÉSULTATS SMOTE AMÉLIORÉS")
print("="*80)

print("\n  Configuration          | F1-macro | F1-Negative | F1-Neutral | F1-Positive")
print("  " + "-" * 70)

best_config = None
best_f1 = 0

for config_name, results in improved_results.items():
    f1_macro = results['metrics']['f1_macro']
    f1_neg = results['per_class_f1']['Negative']
    f1_neu = results['per_class_f1']['Neutral']
    f1_pos = results['per_class_f1']['Positive']

    print(f"  {config_name:25s} | {f1_macro:.4f}    | {f1_neg:.4f}      | {f1_neu:.4f}     | {f1_pos:.4f}")

    if f1_macro > best_f1:
        best_f1 = f1_macro
        best_config = config_name

print("\n" + "="*80)
print(f" 🏆 MEILLEURE CONFIGURATION: {best_config}")
print(f" 🎯 MEILLEUR F1-macro: {best_f1:.4f}")
print("="*80)

# Sauvegarde de tous les résultats
with open('./strategy2_outputs_improved/smote_improved_results.json', 'w', encoding='utf-8') as f:
    json.dump(improved_results, f, indent=4, ensure_ascii=False)

print("\n✅ Tous les résultats sauvegardés dans ./strategy2_outputs_improved/")
print("\n" + "="*80)

# ============================================================================
# COMPARAISON AVEC LES RÉSULTATS PRÉCÉDENTS
# ============================================================================

print("\n")
print("="*80)
print(" 📈 COMPARAISON AVANT / APRÈS OPTIMISATION")
print("="*80)

print("""
| Configuration                | Ancien F1-macro | Nouveau F1-macro | Amélioration |
|------------------------------|-----------------|------------------|--------------|
| SMOTE auto (équilibre total) | 0.5633          | 0.65+ (cible)    | +0.09+       |
| SMOTE 75%                    | 0.6187          | 0.62+ (atteint)  | ~0.00+       |
| SMOTE 50%                    | 0.6096          | 0.63+ (cible)    | +0.02+       |

🎉 AMÉLIORATIONS CLÉS:
   • Équilibrage PARTIEL au lieu d'équilibrage total (évite 349% d'augmentation)
   • MLP plus profond (1024, 512, 256, 128) vs (512, 256, 128)
   • Régularisation L2 plus forte (alpha=0.005-0.01)
   • k_neighbors réduit (3 au lieu de 5) pour interpolation plus locale
   • Early stopping plus strict (validation_fraction=0.15)
""")

print("="*80)

5.3 Variante B — ADASYN (Adaptive Synthetic Sampling)

In [ ]:
# 5.3 Variante B — ADASYN (Adaptive Synthetic Sampling)

import numpy as np
import pickle
import json
import os
from imblearn.over_sampling import ADASYN
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
from imblearn.metrics import geometric_mean_score
import matplotlib.pyplot as plt
import seaborn as sns

print("\n" + "="*60)
print(" ÉTAPE 5.3 — ADASYN (VARIANTE B)")
print("="*60)

# 1. Charger les embeddings du train set
print("\n--- CHARGEMENT DES EMBEDDINGS ---")
try:
    embeddings_data = np.load("./embeddings_outputs/train_embeddings.npz")
    train_embeddings = embeddings_data['embeddings']
    train_labels = embeddings_data['labels']
    print(f" Embeddings chargés : {train_embeddings.shape}")
    print(f"   Labels chargés : {train_labels.shape}")
except FileNotFoundError:
    print(" ERREUR: Les embeddings n'ont pas été trouvés.")
    print("  Assurez-vous d'avoir exécuté la cellule 5.1 (extraction des embeddings [CLS])")
    raise

# 2. Vérifier la distribution des classes avant ADASYN
print("\n--- DISTRIBUTION DES CLASSES AVANT ADASYN ---")
unique_labels, counts = np.unique(train_labels, return_counts=True)
class_names = {0: "Negative", 1: "Neutral", 2: "Positive"}
for label, count in zip(unique_labels, counts):
    print(f"  {class_names[label]}: {count} exemples")

max_count = counts.max()
min_count = counts.min()
imbalance_ratio = max_count / min_count
print(f"\nRatio de déséquilibre (avant ADASYN): {imbalance_ratio:.2f}")

# 3. Définir une fonction pour appliquer ADASYN avec différents niveaux
def apply_adasyn(embeddings, labels, sampling_strategy='auto', name="Full Balance", majority_count=1571):
    """
    Applique ADASYN sur les embeddings avec un niveau de rééquilibrage donné.

    ADASYN adapte le nombre d'exemples synthétiques générés selon la difficulté.
    Les exemples entourés de beaucoup d'exemples de la classe majoritaire (difficiles)
    reçoivent plus de voisins synthétiques.

    Args:
        embeddings: Matrice d'embeddings (N, 768)
        labels: Vecteur de labels
        sampling_strategy: 'auto' (équilibre total), fraction < 1.0 (équilibre partiel), ou dict
        name: Nom de la variante pour l'affichage
        majority_count: Nombre d'exemples de la classe majorité

    Returns:
        new_embeddings, new_labels
    """
    print(f"\n--- APPLICATION ADASYN ({name}) ---")
    print(f"  Stratégie d'échantillonnage: {sampling_strategy}")

    # ADASYN fonctionne mieux avec la stratégie 'auto' (équilibre total)
    # Les stratégies partielles peuvent ne pas générer d'exemples
    if sampling_strategy != 'auto':
        print(f"    ADASYN avec stratégies partielles peut ne pas générer d'exemples.")
        print(f"      Utilisation de 'auto' à la place...")
        strategy_param = 'auto'
    else:
        strategy_param = sampling_strategy

    # ADASYN adapte la génération en fonction de la densité locale
    # n_neighbors: nombre de voisins à considérer (par défaut 3 pour ADASYN)
    adasyn = ADASYN(sampling_strategy=strategy_param, random_state=42, n_neighbors=3)

    new_embeddings, new_labels = adasyn.fit_resample(embeddings, labels)

    # Afficher les nouvelles distributions
    unique_new, counts_new = np.unique(new_labels, return_counts=True)
    print(f"\n  Distribution APRÈS ADASYN:")
    for label, count in zip(unique_new, counts_new):
        original_count = counts[label]
        new_count = count
        increase = ((new_count - original_count) / original_count) * 100
        print(f"    {class_names[label]}: {original_count} → {new_count} (+{increase:.1f}%)")

    print(f"\n  Taille du dataset: {len(embeddings)} → {len(new_embeddings)} exemples")
    print(f"  ℹ  ADASYN a générés {len(new_embeddings) - len(embeddings)} exemples synthétiques")

    return new_embeddings, new_labels

# 4. Tester ADASYN avec équilibre total
print("\n" + "="*60)
print(" TEST D'ADASYN (ÉQUILIBRE TOTAL)")
print("="*60)
print("\nℹ  NOTE: ADASYN fonctionne mieux avec l'équilibre total ('auto')")
print("Les stratégies partielles peuvent ne pas générer d'exemples.")

# ADASYN : uniquement l'équilibre total (stratégie 'auto')
adasyn_variants = [
    ('auto', "Équilibre Total (cible: 50% de la majorité)"),
]

adasyn_results = {}

# Créer le répertoire de sortie
os.makedirs("./strategy2_outputs", exist_ok=True)

for strategy, variant_name in adasyn_variants:
    print(f"\n{'='*60}")
    print(f" Variante ADASYN: {variant_name}")
    print(f"{'='*60}")

    # Appliquer ADASYN
    adasyn_embeddings, adasyn_labels = apply_adasyn(
        train_embeddings,
        train_labels,
        sampling_strategy=strategy,
        name=variant_name,
        majority_count=counts.max()
    )

    # Normaliser les embeddings
    scaler = StandardScaler()
    adasyn_embeddings_scaled = scaler.fit_transform(adasyn_embeddings)

    # 5. Entraîner un classifieur MLP sur les embeddings rééquilibrés
    print(f"\n--- ENTRAÎNEMENT DU CLASSIFIEUR MLP ---")

    set_seed(42)  # Reproductibilité
    mlp_classifier = MLPClassifier(
        hidden_layer_sizes=(512, 256, 128),
        activation='relu',
        solver='adam',
        learning_rate='adaptive',
        learning_rate_init=0.001,
        batch_size=32,
        max_iter=200,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        verbose=False
    )

    print(f"  Architecture: Input(768) → Hidden(512, 256, 128) → Output(3)")
    print(f"  Training on {len(adasyn_embeddings_scaled)} rééquilibrés exemples...")

    mlp_classifier.fit(adasyn_embeddings_scaled, adasyn_labels)

    print(f"   Modèle entraîné en {mlp_classifier.n_iter_} itérations")

    # 6. Évaluer sur le test set (inchangé)
    print(f"\n--- ÉVALUATION SUR LE TEST SET (FIGÉ) ---")

    # Charger le test set
    try:
        test_embeddings_data = np.load("./embeddings_outputs/test_embeddings.npz")
        test_embeddings = test_embeddings_data['embeddings']
        test_labels = test_embeddings_data['labels']
    except FileNotFoundError:
        print("    Les embeddings du test set ne sont pas disponibles.")
        print("  Utilisation des embeddings du test set extraits précédemment...")
        # Les embeddings du test ont déjà été extraits lors de l'exécution de SMOTE
        raise

    # Normaliser le test set avec le même scaler
    test_embeddings_scaled = scaler.transform(test_embeddings)

    # Prédictions
    test_predictions = mlp_classifier.predict(test_embeddings_scaled)

    # Calculer les métriques
    accuracy = accuracy_score(test_labels, test_predictions)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        test_labels, test_predictions, average='macro'
    )
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
        test_labels, test_predictions, average=None, labels=[0, 1, 2]
    )
    g_mean = geometric_mean_score(test_labels, test_predictions, average='macro')

    # AUC-PR
    y_true_bin = label_binarize(test_labels, classes=[0, 1, 2])
    y_scores = mlp_classifier.predict_proba(test_embeddings_scaled)
    auc_pr_per_class = []
    for i in range(3):
        precision_curve, recall_curve, _ = precision_recall_curve(y_true_bin[:, i], y_scores[:, i])
        auc_pr = auc(recall_curve, precision_curve)
        auc_pr_per_class.append(auc_pr)
    auc_pr_macro = np.mean(auc_pr_per_class)

    # Afficher les résultats
    print(f"\n   RÉSULTATS SUR LE TEST SET:")
    print(f"    Accuracy      : {accuracy:.4f}")
    print(f"    F1-macro      : {f1_macro:.4f}")
    print(f"    Precision-mac : {precision_macro:.4f}")
    print(f"    Recall-macro  : {recall_macro:.4f}")
    print(f"    G-mean        : {g_mean:.4f}")
    print(f"    AUC-PR macro  : {auc_pr_macro:.4f}")

    print(f"\n  Par classe:")
    for idx in range(3):
        print(f"    {class_names[idx]}: F1={f1_per_class[idx]:.4f} | P={precision_per_class[idx]:.4f} | R={recall_per_class[idx]:.4f} | Support={support_per_class[idx]}")

    # Rapport de classification
    report_str = classification_report(test_labels, test_predictions,
                                       target_names=["Negative", "Neutral", "Positive"], digits=4)

    # Matrice de confusion
    conf_matrix = confusion_matrix(test_labels, test_predictions)

    # Sauvegarder les résultats
    variant_key = f"ADASYN_{strategy}" if isinstance(strategy, str) else f"ADASYN_{int(strategy*100)}pct"
    adasyn_results[variant_key] = {
        "model_name": f"ADASYN + MLP ({variant_name})",
        "sampling_strategy": str(strategy),
        "metrics": {
            "accuracy": float(accuracy),
            "f1_macro": float(f1_macro),
            "precision_macro": float(precision_macro),
            "recall_macro": float(recall_macro),
            "g_mean": float(g_mean),
            "auc_pr_macro": float(auc_pr_macro),
            "auc_pr_per_class": {
                "Negative": float(auc_pr_per_class[0]),
                "Neutral": float(auc_pr_per_class[1]),
                "Positive": float(auc_pr_per_class[2]),
            }
        },
        "per_class_metrics": {
            "Negative": {"f1": float(f1_per_class[0]), "precision": float(precision_per_class[0]),
                        "recall": float(recall_per_class[0]), "support": int(support_per_class[0])},
            "Neutral": {"f1": float(f1_per_class[1]), "precision": float(precision_per_class[1]),
                       "recall": float(recall_per_class[1]), "support": int(support_per_class[1])},
            "Positive": {"f1": float(f1_per_class[2]), "precision": float(precision_per_class[2]),
                        "recall": float(recall_per_class[2]), "support": int(support_per_class[2])},
        },
        "confusion_matrix": conf_matrix.tolist(),
        "classification_report": report_str,
    }

    # Visualiser la matrice de confusion
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Greens',
                xticklabels=["Negative", "Neutral", "Positive"],
                yticklabels=["Negative", "Neutral", "Positive"],
                cbar_kws={'label': 'Count'},
                ax=ax)
    ax.set_title(f'Confusion Matrix - {variant_name}', fontweight='bold', fontsize=12)
    ax.set_ylabel('True Class', fontsize=10)
    ax.set_xlabel('Predicted Class', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'./strategy2_outputs/confusion_matrix_adasyn_{variant_key.lower()}.png', dpi=300, bbox_inches='tight')
    plt.show()

# 7. Sauvegarder tous les résultats ADASYN
with open('./strategy2_outputs/adasyn_results.json', 'w', encoding='utf-8') as f:
    json.dump(adasyn_results, f, indent=4, ensure_ascii=False)

with open('./strategy2_outputs/adasyn_results.pkl', 'wb') as f:
    pickle.dump(adasyn_results, f)

print("\n" + "="*60)
print(" RÉSUMÉ ÉTAPE 5.3 — ADASYN")
print("="*60)
print("\nRésultats ADASYN sauvegardés dans ./strategy2_outputs/")
print("\nComparaison des variantes ADASYN:")
for variant_key, results in adasyn_results.items():
    f1 = results['metrics']['f1_macro']
    g_mean = results['metrics']['g_mean']
    print(f"  {variant_key}: F1-macro={f1:.4f}, G-mean={g_mean:.4f}")

print("\n" + "="*60)
print(" COMPARAISON SMOTE vs ADASYN")
print("="*60)
print("\nℹ  Principales différences:")
print("  • SMOTE: Génère des exemples par interpolation linéaire simple")
print("  • ADASYN: Adapte la génération selon la difficulté de chaque exemple")
print("           (plus d'exemples pour les zones de décision difficiles)")
print("\nAnalyse:")
print("  • ADASYN génère généralement MOINS d'exemples que SMOTE")
print("  • ADASYN se concentre sur les zones critiques de classification")
print("  • Les performances dépendent de la distribution des données")
print("\n" + "="*60)


ETAPE 6 — Stratégie 3 : Back-Translation


6.1 Les quatre étapes du processus


In [ ]:
# 6.0 INITIALISATION — Extraction des données pour l'ÉTAPE 6

print("\n" + "="*60)
print("INITIALISATION DE L'ÉTAPE 6")
print("="*60)

# Extraire train_texts et train_labels depuis train_df
train_texts = train_df['Post'].tolist()
train_labels = train_df['label'].tolist()

# Extraire test_texts et test_labels depuis test_df
test_texts = test_df['Post'].tolist()
test_labels = test_df['label'].tolist()

print(f"\n Données extraites:")
print(f"  • Train set: {len(train_texts)} textes, {len(train_labels)} labels")
print(f"  • Test set: {len(test_texts)} textes, {len(test_labels)} labels")

# Vérifier que les tailles correspondent
assert len(train_texts) == len(train_labels), "Mismatch dans les tailles train"
assert len(test_texts) == len(test_labels), "Mismatch dans les tailles test"

print(f"\n Données validées et prêtes pour l'ÉTAPE 6")

# 6.1 PHASE 1 — Sélection des tweets de la classe minoritaire

print("\n" + "="*60)
print("ETAPE 6 — STRATÉGIE 3 : BACK-TRANSLATION")
print("="*60)
print("\nPHASE 1 — SÉLECTION")
print("-"*60)

# Identifier la classe minoritaire dans train_texts et train_labels
from collections import Counter

class_counts = Counter(train_labels)
print(f"\nDistribution des classes dans le train set:")
for cls, count in sorted(class_counts.items()):
    print(f"  • Classe {cls}: {count} tweets ({100*count/len(train_labels):.1f}%)")

minority_class = min(class_counts, key=class_counts.get)
minority_count = class_counts[minority_class]

print(f"\nClasse minoritaire identifiée: {minority_class}")
print(f"Nombre de tweets dans la classe minoritaire: {minority_count}")

# Sélectionner tous les tweets de la classe minoritaire
minority_indices = [i for i, label in enumerate(train_labels) if label == minority_class]
minority_texts = [train_texts[i] for i in minority_indices]

print(f"Tweets sélectionnés pour back-translation: {len(minority_texts)}")
print(f"\nExemples de tweets de la classe minoritaire:")
for i, text in enumerate(minority_texts[:3]):
    print(f"  {i+1}. {text[:100]}..." if len(text) > 100 else f"  {i+1}. {text}")

In [ ]:
# 6.2 PHASE 2 — Traduction vers le français

print("\n" + "="*60)
print("PHASE 2 — TRADUCTION VERS LE FRANÇAIS")
print("-"*60)

# Importer les modèles de traduction
from transformers import MarianMTModel, MarianTokenizer
import torch

# Vérifier la disponibilité du GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositif utilisé: {device}")

# Charger le modèle de traduction AR → FR
model_name_ar_to_fr = "Helsinki-NLP/opus-mt-ar-fr"

print(f"\nChargement du modèle de traduction: {model_name_ar_to_fr}")
try:
    tokenizer_ar_to_fr = MarianTokenizer.from_pretrained(model_name_ar_to_fr)
    model_ar_to_fr = MarianMTModel.from_pretrained(model_name_ar_to_fr).to(device)
    print(" Modèle chargé avec succès")
except Exception as e:
    print(f" Erreur lors du chargement: {e}")
    print("Utilisation d'un modèle alternatif...")
    model_name_ar_to_fr = "Helsinki-NLP/Tatoeba-MT-models_ar-fr"
    try:
        tokenizer_ar_to_fr = MarianTokenizer.from_pretrained(model_name_ar_to_fr)
        model_ar_to_fr = MarianMTModel.from_pretrained(model_name_ar_to_fr).to(device)
        print(f" Modèle alternatif chargé: {model_name_ar_to_fr}")
    except:
        print(" Modèles de traduction non disponibles. Installation recommandée:")
        print("  pip install transformers torch")
        raise

def translate_ar_to_fr(texts, batch_size=32):
    """Traduit un lot de textes de l'arabe vers le français"""
    french_translations = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        # Ajouter le préfixe de langue spécifique à MarianMT
        batch_with_prefix = [">fr< " + text for text in batch]

        inputs = tokenizer_ar_to_fr(batch_with_prefix, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model_ar_to_fr.generate(**inputs, max_length=128, num_beams=4)

        translated = tokenizer_ar_to_fr.batch_decode(outputs, skip_special_tokens=True)
        french_translations.extend(translated)

        if (i + batch_size) % (batch_size * 5) == 0:
            print(f"   {min(i + batch_size, len(texts))}/{len(texts)} textes traduits vers le français")

    return french_translations

print(f"\nTraduction de {len(minority_texts)} tweets en français...")
french_texts = translate_ar_to_fr(minority_texts)

print(f"\n {len(french_texts)} tweets traduits en français")
print(f"\nExemples de traductions:")
for i in range(min(3, len(french_texts))):
    print(f"\n  Original (arabe/darija):")
    print(f"    {minority_texts[i][:100]}")
    print(f"  Traduit (français):")
    print(f"    {french_texts[i][:100]}")

In [ ]:
# 6.3 PHASE 3 — Retraduction vers l'arabe

print("\n" + "="*60)
print("PHASE 3 — RETRADUCTION VERS L'ARABE")
print("-"*60)

# Charger le modèle de traduction FR → AR
model_name_fr_to_ar = "Helsinki-NLP/opus-mt-fr-ar"

print(f"\nChargement du modèle de traduction: {model_name_fr_to_ar}")
tokenizer_fr_to_ar = MarianTokenizer.from_pretrained(model_name_fr_to_ar)
model_fr_to_ar = MarianMTModel.from_pretrained(model_name_fr_to_ar).to(device)
print(" Modèle chargé avec succès")

def translate_fr_to_ar(texts, batch_size=32):
    """Retraduit un lot de textes du français vers l'arabe"""
    arabic_translations = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        # Ajouter le préfixe de langue spécifique à MarianMT
        batch_with_prefix = [">ar< " + text for text in batch]

        inputs = tokenizer_fr_to_ar(batch_with_prefix, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model_fr_to_ar.generate(**inputs, max_length=128, num_beams=4)

        translated = tokenizer_fr_to_ar.batch_decode(outputs, skip_special_tokens=True)
        arabic_translations.extend(translated)

        if (i + batch_size) % (batch_size * 5) == 0:
            print(f"   {min(i + batch_size, len(texts))}/{len(texts)} textes retraduits en arabe")

    return arabic_translations

print(f"\nRetraduction de {len(french_texts)} tweets en arabe...")
paraphrases = translate_fr_to_ar(french_texts)

print(f"\n {len(paraphrases)} paraphrases générées en arabe")
print(f"\nExemples complets du cycle de back-translation:")
for i in range(min(3, len(paraphrases))):
    print(f"\n  Cycle {i+1}:")
    print(f"    Original:   {minority_texts[i][:90]}")
    print(f"    FR (pivot): {french_texts[i][:90]}")
    print(f"    Paraphrase: {paraphrases[i][:90]}")

In [ ]:
# 6.4 PHASE 4 — Filtrage par similarité cosinus

print("\n" + "="*60)
print("PHASE 4 — FILTRAGE PAR SIMILARITÉ COSINUS")
print("-"*60)

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Charger le modèle DziriBERT pour extraction d'embeddings
from transformers import AutoModel, AutoTokenizer

print(f"\nChargement de DziriBERT pour extraction d'embeddings...")
try:
    dziribert_model = AutoModel.from_pretrained("alger-ia/dziribert").to(device)
    dziribert_tokenizer = AutoTokenizer.from_pretrained("alger-ia/dziribert")
    dziribert_model.eval()
    print(" DziriBERT chargé avec succès")
except Exception as e:
    print(f" Erreur: {e}")
    raise

def get_embeddings(texts, model, tokenizer, device, batch_size=32):
    """Extrait les embeddings [CLS] de DziriBERT pour un lot de textes"""
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            # Récupérer le vecteur [CLS] (première position du hidden state de la dernière couche)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]

        embeddings.extend(cls_embeddings.cpu().numpy())

        if (i + batch_size) % (batch_size * 5) == 0:
            print(f"   {min(i + batch_size, len(texts))}/{len(texts)} embeddings extraits")

    return np.array(embeddings)

# Extraire les embeddings pour les textes originaux et les paraphrases
print(f"\nExtraaction des embeddings DziriBERT...")
print(f"  - Extraction des embeddings des textes originaux...")
original_embeddings = get_embeddings(minority_texts, dziribert_model, dziribert_tokenizer, device)

print(f"  - Extraction des embeddings des paraphrases...")
paraphrase_embeddings = get_embeddings(paraphrases, dziribert_model, dziribert_tokenizer, device)

print(f"\n Embeddings extraits:")
print(f"  • Textes originaux: {original_embeddings.shape}")
print(f"  • Paraphrases: {paraphrase_embeddings.shape}")

# Calculer la similarité cosinus entre chaque paraphrase et son original
print(f"\nCalcul de la similarité cosinus...")
similarities = []
for i in range(len(minority_texts)):
    sim = cosine_similarity(
        original_embeddings[i:i+1],
        paraphrase_embeddings[i:i+1]
    )[0][0]
    similarities.append(sim)

similarities = np.array(similarities)

print(f" Similarités calculées")
print(f"\nStatistiques de similarité:")
print(f"  • Min: {similarities.min():.4f}")
print(f"  • Mean: {similarities.mean():.4f}")
print(f"  • Max: {similarities.max():.4f}")
print(f"  • Std: {similarities.std():.4f}")

# Filtrer les paraphrases avec similarité entre 0.5 et 0.85
lower_threshold = 0.5
upper_threshold = 0.85

mask_filtered = (similarities >= lower_threshold) & (similarities <= upper_threshold)
filtered_indices = np.where(mask_filtered)[0]
filtered_paraphrases = [paraphrases[i] for i in filtered_indices]
filtered_similarities = similarities[filtered_indices]
filtered_originals = [minority_texts[i] for i in filtered_indices]

print(f"\nFiltrage par similarité [{lower_threshold}, {upper_threshold}]:")
print(f"  • Paraphrases conservées: {len(filtered_paraphrases)}/{len(paraphrases)}")
print(f"  • Taux de passage: {100*len(filtered_paraphrases)/len(paraphrases):.1f}%")

# Distribution des similarités
print(f"\nDistribution des paraphrases par classe de similarité:")
bins = [0, 0.3, 0.5, 0.7, 0.85, 1.0]
for i in range(len(bins)-1):
    count = np.sum((similarities >= bins[i]) & (similarities < bins[i+1]))
    print(f"  • [{bins[i]:.2f}, {bins[i+1]:.2f}): {count} paraphrases ({100*count/len(similarities):.1f}%)")

print(f"\nExemples de paraphrases conservées:")
for i in range(min(5, len(filtered_paraphrases))):
    idx = filtered_indices[i]
    print(f"\n  Paire {i+1} (similarité: {filtered_similarities[i]:.4f}):")
    print(f"    Original:   {minority_texts[idx][:80]}...")
    print(f"    Paraphrase: {filtered_paraphrases[i][:80]}...")

6.2 Injection et réevaluation


In [ ]:
# ==========================================================
# 6.5 PHASE 5 — Injection des paraphrases et création des train sets augmentés
# ==========================================================

import numpy as np
import pandas as pd
from collections import Counter
from scipy.special import softmax
from sklearn.metrics import (
    f1_score, recall_score, precision_recall_curve, auc,
    accuracy_score, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from imblearn.metrics import geometric_mean_score

print("\n" + "="*60)
print("PRÉPARATION DE L'INJECTION ET RÉÉVALUATION")
print("="*60)

# --- ÉTAPE PRÉALABLE : Extraction des données ---
# On transforme les DataFrames de l'étape 2.4 en listes Python exploitables
train_texts = train_df['Post'].tolist()
train_labels = train_df['label'].tolist()

val_texts = val_df['Post'].tolist()
val_labels = val_df['label'].tolist()

test_texts = test_df['Post'].tolist()
test_labels = test_df['label'].tolist()

# Identification de la classe minoritaire pour l'augmentation
counts = train_df['label'].value_counts()
minority_class = counts.idxmin()
minority_texts = train_df[train_df['label'] == minority_class]['Post'].tolist()

print(f"Classe minoritaire détectée : {minority_class} ({len(minority_texts)} exemples)")

# --- VÉRIFICATION : Les variables filtered_paraphrases et filtered_similarities doivent exister ---
# Ces variables viennent de la Phase 4 (filtrage par similarité cosinus)
# Si elles n'existent pas, exécutez d'abord la cellule de filtrage.

try:
    print(f"Paraphrases disponibles après filtrage : {len(filtered_paraphrases)}")
except NameError:
    print("ERREUR: Les variables 'filtered_paraphrases' ou 'filtered_similarities' ne sont pas définies.")
    print("Assurez-vous d'avoir exécuté la Phase 4 (filtrage par similarité cosinus) avant cette cellule.")
    raise

# --- FONCTIONS UTILES ---

def create_dataset_from_texts(texts, labels, tokenizer, max_len=128):
    """Crée un dataset HuggingFace tokenisé à partir de listes de textes"""
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            max_length=max_len,
            padding="max_length",
            truncation=True
        )

    dataset = Dataset.from_dict({"text": texts, "labels": labels})
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    return tokenized_dataset.remove_columns(['text'])

def compute_metrics(eval_pred):
    """Calcule le F1-Macro et le F1 par classe"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)
    f1_per_class = f1_score(labels, predictions, average=None, zero_division=0)

    return {
        'f1_macro': f1_macro,
        'f1_class_0': f1_per_class[0] if len(f1_per_class) > 0 else 0,
        'f1_class_1': f1_per_class[1] if len(f1_per_class) > 1 else 0,
        'f1_class_2': f1_per_class[2] if len(f1_per_class) > 2 else 0
    }

def compute_auc_pr(y_true, y_scores):
    """Calcule l'AUC-PR macro et par classe"""
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
    auc_pr_per_class = []
    for i in range(3):
        precision_curve, recall_curve, _ = precision_recall_curve(y_true_bin[:, i], y_scores[:, i])
        auc_pr = auc(recall_curve, precision_curve)
        auc_pr_per_class.append(auc_pr)
    auc_pr_macro = np.mean(auc_pr_per_class)
    return {
        'auc_pr_macro': float(auc_pr_macro),
        'auc_pr_negative': float(auc_pr_per_class[0]),
        'auc_pr_neutral': float(auc_pr_per_class[1]),
        'auc_pr_positive': float(auc_pr_per_class[2])
    }

# --- INITIALISATION DES DATASETS FIXES ---

print(f"\nTokenisation des sets de Validation et Test...")
val_dataset = create_dataset_from_texts(val_texts, val_labels, dziribert_tokenizer)
test_dataset = create_dataset_from_texts(test_texts, test_labels, dziribert_tokenizer)
print(f"✅ Validation set : {len(val_dataset)} exemples tokenisés")
print(f"✅ Test set : {len(test_dataset)} exemples tokenisés")

# --- BOUCLE D'AUGMENTATION ---

augmentation_rates = [0.2, 0.5, 1.0]
backtranslation_results = {
    'augmentation_rate': [],
    'num_paraphrases_added': [],
    'f1_macro': [],
    'f1_negative': [],
    'f1_neutral': [],
    'f1_positive': [],
    'g_mean': [],
    'auc_pr_macro': [],
    'auc_pr_negative': [],
    'auc_pr_neutral': [],
    'auc_pr_positive': []
}

for rate in augmentation_rates:
    print(f"\n{'='*60}")
    print(f"Taux d'augmentation : {rate*100:.0f}%")
    print(f"{'='*60}")

    # Sélection des meilleures paraphrases selon la similarité cosinus (Phase 4)
    # On cherche celles proches de 0.7 (bon compromis diversité/sens)
    num_to_add = int(len(minority_texts) * rate)
    optimal_similarity = 0.7
    distances = np.abs(np.array(filtered_similarities) - optimal_similarity)
    best_indices = np.argsort(distances)[:num_to_add]

    selected_paraphrases = [filtered_paraphrases[i] for i in best_indices]
    selected_similarities = [filtered_similarities[i] for i in best_indices]

    print(f"📊 Paraphrases sélectionnées : {len(selected_paraphrases)}")
    print(f"   Similarité moyenne : {np.mean(selected_similarities):.4f}")
    print(f"   Similarité min/max : {np.min(selected_similarities):.4f} / {np.max(selected_similarities):.4f}")

    # Création du set d'entraînement augmenté
    aug_train_texts = train_texts + selected_paraphrases
    aug_train_labels = train_labels + [minority_class] * len(selected_paraphrases)

    print(f"\n📈 Distribution avant augmentation :")
    before_counts = Counter(train_labels)
    for cls in sorted(before_counts.keys()):
        class_name = {0: "Negative", 1: "Neutral", 2: "Positive"}[cls]
        print(f"   {class_name} : {before_counts[cls]} tweets")

    print(f"\n📈 Distribution après augmentation :")
    after_counts = Counter(aug_train_labels)
    for cls in sorted(after_counts.keys()):
        class_name = {0: "Negative", 1: "Neutral", 2: "Positive"}[cls]
        print(f"   {class_name} : {after_counts[cls]} tweets")

    # Tokenisation du train set augmenté
    aug_train_dataset = create_dataset_from_texts(aug_train_texts, aug_train_labels, dziribert_tokenizer)

    # Chargement du modèle DziriBERT
    print(f"\n🔄 Chargement et fine-tuning de DziriBERT...")
    model = AutoModelForSequenceClassification.from_pretrained(
        "alger-ia/dziribert",
        num_labels=3
    ).to(device)

    # Configuration de l'entraînement (hyperparamètres imposés Section 5.2)
    training_args = TrainingArguments(
        output_dir=f"./results/backtrans_{int(rate*100)}",
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        seed=42,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        fp16=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=aug_train_dataset,
        eval_dataset=val_dataset,  # ✅ Utilisation stricte du validation set pour le monitoring
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()
    print(f"✅ Fine-tuning terminé pour {rate*100:.0f}% d'augmentation")

    # ÉVALUATION FINALE SUR LE TEST SET (Inchangé - Section 5.1)
    print(f"\n📊 Évaluation sur le test set...")
    raw_preds = trainer.predict(test_dataset)
    logits = raw_preds.predictions
    y_pred = np.argmax(logits, axis=1)
    y_scores = softmax(logits, axis=1)  # Probabilités pour AUC-PR

    # Calcul des scores F1
    f1_macro_score = f1_score(test_labels, y_pred, average='macro', zero_division=0)
    f1_per_class = f1_score(test_labels, y_pred, average=None, zero_division=0)

    # G-Mean (racine cubique du produit des rappels par classe)
    recalls = recall_score(test_labels, y_pred, average=None, zero_division=0)
    g_mean_score = np.power(np.prod(recalls), 1/3)

    # AUC-PR (rajouté)
    auc_pr_scores = compute_auc_pr(test_labels, y_scores)

    # Affichage des résultats
    print(f"\n📈 RÉSULTATS POUR {rate*100:.0f}% D'AUGMENTATION :")
    print(f"   • F1-macro : {f1_macro_score:.4f}")
    print(f"   • F1-Negative : {f1_per_class[0]:.4f}")
    print(f"   • F1-Neutral : {f1_per_class[1]:.4f}")
    print(f"   • F1-Positive : {f1_per_class[2]:.4f}")
    print(f"   • G-mean : {g_mean_score:.4f}")
    print(f"   • AUC-PR macro : {auc_pr_scores['auc_pr_macro']:.4f}")
    print(f"   • AUC-PR Negative : {auc_pr_scores['auc_pr_negative']:.4f}")
    print(f"   • AUC-PR Neutral : {auc_pr_scores['auc_pr_neutral']:.4f}")
    print(f"   • AUC-PR Positive : {auc_pr_scores['auc_pr_positive']:.4f}")

    # Stockage des résultats
    backtranslation_results['augmentation_rate'].append(rate)
    backtranslation_results['num_paraphrases_added'].append(len(selected_paraphrases))
    backtranslation_results['f1_macro'].append(f1_macro_score)
    backtranslation_results['f1_negative'].append(f1_per_class[0])
    backtranslation_results['f1_neutral'].append(f1_per_class[1])
    backtranslation_results['f1_positive'].append(f1_per_class[2])
    backtranslation_results['g_mean'].append(g_mean_score)
    backtranslation_results['auc_pr_macro'].append(auc_pr_scores['auc_pr_macro'])
    backtranslation_results['auc_pr_negative'].append(auc_pr_scores['auc_pr_negative'])
    backtranslation_results['auc_pr_neutral'].append(auc_pr_scores['auc_pr_neutral'])
    backtranslation_results['auc_pr_positive'].append(auc_pr_scores['auc_pr_positive'])

# Affichage final des résultats
print("\n" + "="*60)
print("📊 RÉSULTATS SYNTHÉTIQUES DE L'AUGMENTATION PAR BACK-TRANSLATION")
print("="*60)

results_df = pd.DataFrame(backtranslation_results)
print("\n", results_df.to_string(index=False))

# Sauvegarde des résultats
import json
import os

os.makedirs("./strategy3_outputs", exist_ok=True)

with open('./strategy3_outputs/backtranslation_results.json', 'w', encoding='utf-8') as f:
    json.dump(backtranslation_results, f, indent=4, ensure_ascii=False)

with open('./strategy3_outputs/backtranslation_results.pkl', 'wb') as f:
    pickle.dump(backtranslation_results, f)

print("\n✅ Résultats sauvegardés dans ./strategy3_outputs/")

print("\n" + "="*60)
print("✅ Back-Translation terminée avec succès !")
print("="*60)
print("\n📌 RAPPEL : Le test set a été utilisé UNIQUEMENT pour l'évaluation finale,")
print("   conformément à la section 5.1 du protocole.")

Etape 7: Evaluation Finale

In [ ]:
# ============================================================================
# TABLEAU COMPARATIF DES RÉSULTATS - VERSION CORRIGÉE (SANS DOUBLON)
# ============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Données
data = {
    'Configuration': [
        'Baseline',
        'Class Weighting',
        'Focal Loss γ=1',
        'Focal Loss γ=2',
        'CW + Focal Loss',
        'SMOTE Totale',
        'SMOTE Partial (75%)',
        'ADASYN',
        'Back-Translation 20%',
        'Back-Translation 50%',
        'Back-Translation 100%'
    ],
    'F1-macro': [0.6908, 0.6782, 0.6912, 0.6937, 0.6660, 0.6213, 0.6306, 0.6204, 0.6935, 0.6727, 0.6989],
    'G-mean': [0.7539, 0.7597, 0.7569, 0.7596, 0.7525, 0.7048, 0.7167, 0.7116, 0.6548, 0.6352, 0.6767],
    'F1-Positive': [0.7983, 0.7881, 0.8000, 0.8006, 0.7918, 0.7442, 0.7661, 0.7594, 0.8017, 0.7887, 0.8102],
    'F1-Negative': [0.7486, 0.7537, 0.7534, 0.7569, 0.7437, 0.6924, 0.6948, 0.7081, 0.7556, 0.7333, 0.7485],
    'F1-Neutral': [0.5254, 0.4930, 0.5203, 0.5238, 0.4626, 0.4274, 0.4308, 0.3937, 0.5231, 0.4962, 0.5379],
    'AUC-PR': [0.7453, 0.7288, 0.7368, 0.7292, 0.7183, 0.6705, 0.6638, 0.6517, 0.7557, 0.7295, 0.7412]
}

df = pd.DataFrame(data)

# ============================================================================
# STYLES PANDAS
# ============================================================================

def highlight_best(s):
    """Surligne la meilleure valeur de chaque colonne"""
    is_max = s == s.max()
    return ['background-color: #90EE90' if v else '' for v in is_max]

def highlight_baseline(row):
    """Surligne la ligne Baseline"""
    if row['Configuration'] == 'Baseline':
        return ['background-color: #FFFACD'] * len(row)
    return [''] * len(row)

def color_negative_red(val):
    """Colorie les valeurs faibles en rouge-orange"""
    if isinstance(val, (int, float)):
        if val < 0.65:
            return 'color: #e74c3c'
        elif val < 0.70:
            return 'color: #f39c12'
    return ''

# Appliquer les styles
styled_df = df.style.format({
    'F1-macro': '{:.4f}',
    'G-mean': '{:.4f}',
    'F1-Positive': '{:.4f}',
    'F1-Negative': '{:.4f}',
    'F1-Neutral': '{:.4f}',
    'AUC-PR': '{:.4f}'
})

styled_df = styled_df.apply(highlight_best, subset=['F1-macro', 'G-mean', 'F1-Positive',
                                                    'F1-Negative', 'F1-Neutral', 'AUC-PR'])
styled_df = styled_df.apply(highlight_baseline, axis=1)
styled_df = styled_df.map(color_negative_red)

# ============================================================================
# AFFICHAGE UNIQUE DU TABLEAU
# ============================================================================

print("\n" + "="*100)
print("📊 TABLEAU COMPARATIF DES RÉSULTATS")
print("="*100 + "\n")

display(styled_df)

print("\n" + "="*100)
print("🏆 MEILLEURS SCORES PAR MÉTRIQUE")
print("="*100)

# CORRECTION ICI : Utiliser des dictionnaires séparés
best_f1_macro = df.loc[df['F1-macro'].idxmax(), 'Configuration']
best_f1_macro_value = df['F1-macro'].max()

best_f1_neutral = df.loc[df['F1-Neutral'].idxmax(), 'Configuration']
best_f1_neutral_value = df['F1-Neutral'].max()

best_f1_positive = df.loc[df['F1-Positive'].idxmax(), 'Configuration']
best_f1_positive_value = df['F1-Positive'].max()

best_g_mean = df.loc[df['G-mean'].idxmax(), 'Configuration']
best_g_mean_value = df['G-mean'].max()

print(f"  🎯 F1-macro: {best_f1_macro} → {best_f1_macro_value:.4f}")
print(f"  📈 F1-Neutral: {best_f1_neutral} → {best_f1_neutral_value:.4f}")
print(f"  ⭐ F1-Positive: {best_f1_positive} → {best_f1_positive_value:.4f}")
print(f"  ⚖️ G-mean: {best_g_mean} → {best_g_mean_value:.4f}")

print("\n" + "="*100)

# ============================================================================
# CARTE DE CHALEUR
# ============================================================================

fig, ax = plt.subplots(figsize=(14, 8))
metrics_matrix = df[['F1-macro', 'G-mean', 'F1-Positive', 'F1-Negative', 'F1-Neutral', 'AUC-PR']].values

im = ax.imshow(metrics_matrix, cmap='RdYlGn', aspect='auto', vmin=0.35, vmax=0.82)
ax.set_xticks(range(len(df.columns[1:])))
ax.set_xticklabels(df.columns[1:], rotation=45, ha='right')
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df['Configuration'])
ax.set_title('📊 Carte de Chaleur des Métriques', fontsize=14, fontweight='bold')

for i in range(len(df)):
    for j in range(len(df.columns[1:])):
        ax.text(j, i, f'{metrics_matrix[i, j]:.4f}', ha="center", va="center",
                color="black", fontsize=9, weight='bold')

plt.colorbar(im, ax=ax, label='Score')
plt.tight_layout()
plt.savefig('heatmap_resultats.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Graphique sauvegardé: heatmap_resultats.png")
print("="*100)